In [ ]:
%reset -f
import math
import os
import warnings
from collections.abc import Callable
from math import pow, sqrt

import forallpeople as units
import handcalcs
import handcalcs.render
import numpy as np

# import polars as pl
import pandas as pd
import sympy as sp
from structural_tools.notebook.calculation import (
    check_value,
    set_params_columns,
)
from structural_tools.notebook.display import (
    display_table,
    display_text,
)
from structural_tools.seismic.asce.parameters import LOAD_FACTOR_SEISMIC_ASD, RiskCategory, SeismicParameters, SiteClass

units.environment(env_name="structural", top_level=True)
handcalcs.set_option("custom_symbols", {"star": "^*", "__": ",", "plus": "+", "minus": "-"})
handcalcs.set_option("greek_exclusions", ["psi"])
handcalcs.set_option("display_precision", 4)
handcalcs.set_option("param_columns", 1)
handcalcs.set_option("latex_block_start", "\\begin{equation*}")
handcalcs.set_option("latex_block_end", "\\end{equation*}")

# Effective Seismic Weight Calculation

We calculate the effective seismic weight with unit weights and square footages.

## Unit Weights

Weights will be assigned with area unit weights.

In [3]:
set_params_columns(3)

In [4]:
%%render params
U_floor = 35 / 1000 * ksf
U_roof = 25 / 1000 * ksf
U_wall = 12 / 1000 * ksf

<IPython.core.display.Latex object>

## Building Geometry

The lengths and widths of the weight areas are consistent across levels 2, 3, 4, and 5. Lengths, $L$, are the E-W dimensions, while widths, $w$, are the N-S dimensions.

Due to the building geometry, the floor levels are split into different zones shown in ![Building area labels](images/building_e_areas.png){width=60% #fig:diaphragm_trib_area}

In [5]:
set_params_columns(4)

In [6]:
%%render params
L_1 = 131 + (10 / 12) * ft
L_2 = 170 + (2 / 12) * ft
L_3 = 36 + (3 / 12) * ft
L_4 = 30 + (9 / 12) * ft
w_1 = 36 + (3 / 12) * ft
w_2 = 30 + (9 / 12) * ft
w_3 = 115 + (1 / 12) * ft
w_4 = 120 + (1 / 12) * ft
L_jog = 5 * ft

<IPython.core.display.Latex object>

The wall heights vary across the levels.

In [7]:
set_params_columns(5)

In [8]:
%%render params
h_wall__1 = 11.5 * ft
h_wall__2 = 10.5 * ft
h_wall__3 = 10.5 * ft
h_wall__4 = 10.5 * ft
h_wall__5 = 10 * ft

<IPython.core.display.Latex object>

The perimeter of the building is:

> Note: there is a small overlap of zones 2 and 4 between lines E-9 and E-10

In [9]:
Overlap = (6 + 7 / 12) * ft

In [10]:
%%render
P_wall = L_1 + w_1 + L_jog + w_2 + L_2 + w_2 - Overlap + L_4 + w_4 + L_4 + L_jog + L_3 + w_3 - w_1

<IPython.core.display.Latex object>

Areas can now be calculated. Note that the areas of the walls are the actual wall areas; they have not been distributed to each floor based on tributary area, yet.

In [11]:
%%render long
A_level = L_1 * w_1 + L_2 * w_2 + L_3 * w_3 + L_4 * w_4

<IPython.core.display.Latex object>

In [12]:
%%render
A_wall__1 = h_wall__1 * P_wall
A_wall__2 = h_wall__2 * P_wall
A_wall__3 = h_wall__3 * P_wall
A_wall__4 = h_wall__4 * P_wall
A_wall__5 = h_wall__5 * P_wall

<IPython.core.display.Latex object>

## Component Weights

The typical floor and corridor weights are

In [13]:
%%render
W_level = U_floor * A_level

<IPython.core.display.Latex object>

This applies to levels 2, 3, and 4.

In [14]:
set_params_columns(4)

In [15]:
%%render params
W_level__2 = W_level
W_level__3 = W_level
W_level__4 = W_level
W_level__5 = W_level

<IPython.core.display.Latex object>

The roof weight is

In [16]:
%%render
W_roof = U_roof * A_level

<IPython.core.display.Latex object>

The exterior walls are divided in half to the floor above and below.

In [17]:
%%render
W_wall__2 = U_wall * (A_wall__2 / 2 + A_wall__1 / 2)
W_wall__3 = U_wall * (A_wall__3 / 2 + A_wall__2 / 2)
W_wall__4 = U_wall * (A_wall__4 / 2 + A_wall__3 / 2)
W_wall__5 = U_wall * (A_wall__5 / 2 + A_wall__4 / 2)
W_wall__roof = U_wall * (A_wall__5 / 2)

<IPython.core.display.Latex object>

## Level Weights

Each level's weight is

In [18]:
%%render 
W_level__2 = W_level__2 + W_wall__2
W_level__3 = W_level__3 + W_wall__3
W_level__4 = W_level__4 + W_wall__4
W_level__5 = W_level__5 + W_wall__5
# be sure to add the weight of the truss later
W_level__roof = W_roof + W_wall__roof

<IPython.core.display.Latex object>

## Total Weight

The total weight of the structure is

In [19]:
%%render long
W_total = W_level__2 + W_level__3 + W_level__4 + W_level__5 + W_level__roof

<IPython.core.display.Latex object>

# Seismic Equivalent Lateral Force (ELF) Analysis

We will be using the equivalent lateral force method prescribed by ASCE 7-16 Section 12.8. We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

## Seismic Parameters

In [20]:
S_S = 0.887  # g
S_1 = 0.410  # g
seismic_params = SeismicParameters(s_1=S_1, s_s=S_S, risk_category=RiskCategory.II, site_class=SiteClass.D)

Seismic design parameters are obtained from the geotech report and ASCE 7-16 Sec. 11 and 12.

In [21]:
set_params_columns(5)

In [22]:
%%render params
R = 6.5
Omega_0 = 3.0
C_d = 4.0
rho = 1.0

<IPython.core.display.Latex object>

In [23]:
%%render params
I_e = seismic_params.i_e
S_S = seismic_params.s_s  # g
S_1 = seismic_params.s_1  # g
F_a = seismic_params.f_a
F_v = seismic_params.f_v
S_MS = seismic_params.s_ms  # g
S_M1 = seismic_params.s_m1  # g
S_DS = seismic_params.s_ds  # g
S_D1 = seismic_params.s_d1  # g
SDC = seismic_params.sdc
C_u = 1.4
T_L = 16  # s

<IPython.core.display.Latex object>

## Base Shear

We must calculate the period

In [24]:
set_params_columns(1)

In [25]:
%%render params
h_top__roof = (68 + 3 / 12) * ft
h_bottom__roof = (55 + 10 / 12) * ft
h_n = np.mean([h_top__roof, h_bottom__roof])  # ASCE 7-16 Sec. 11.2

<IPython.core.display.Latex object>

In [26]:
%%render
h_n__check = check_value(h_n, 65 * ft, "<=")

<IPython.core.display.Latex object>

We will approximate the building period.

In [27]:
h_n = float(h_n)  # to prevent weirdness with units

In [28]:
%%render
C_t = 0.02  # ASCE 7-16 Table 12.8-2
x = 0.75  # ASCE 7-16 Table 12.8-2
T_a = C_t * h_n**x  # ASCE 7-16 Eq. 12.8-7

<IPython.core.display.Latex object>

Because we are not using an automated analysis software, we can make the following assumption:

In [29]:
%%render
T = T_a  # ASCE 7-16 Sec. 12.8.2

<IPython.core.display.Latex object>

With the period, we can calculate the seismic response coefficient.

In [30]:
%%render
C_s = S_DS / (R / I_e)  # ASCE 7-16 Eq. 12.8-2

<IPython.core.display.Latex object>

We must check that $C_s$ does not exceed:

In [31]:
%%render long
# fmt: off
if T <= T_L:  C_s__check__max = S_D1 / (T * (R / I_e));              C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
elif T > T_L: C_s__check__max = (S_D1 * T_L) / (T**2 * (R / I_e));   C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
# fmt: on

<IPython.core.display.Latex object>

And we must also check that $C_s$ is not less than:

In [32]:
%%render
C_s__check__min__1 = 0.044 * S_DS * I_e  # ASCE 7-16 Eq. 12.8-5
C_s__check__min = check_value(C_s, C_s__check__min__1, "geq")

<IPython.core.display.Latex object>

as well as:

In [33]:
%%render
C_s__check__min = check_value(C_s, 0.01, "geq")  # ASCE 7-16 Eq. 12.8-5

<IPython.core.display.Latex object>

and finally:

In [34]:
%%render
# fmt: off
if S_1 >= 0.6:  C_s__check__min__3 = 0.5 * S_1 / (R / I_e); C_s__check__min = check_value(C_s, C_s__check__min__3, "geq")  # ASCE 7-16 Eq. 12.8-6
elif S_1 < 0.6: C_s__check__min = "\\text{skipping check}"  # ASCE 7-16 Eq. 12.8-6
# fmt: on

<IPython.core.display.Latex object>

And now, we can calculate the base shear, $V$

In [35]:
%%render
V = C_s * W_total  # ASCE 7-16 Eq. 12.8-1

<IPython.core.display.Latex object>

## Distribute the seismic forces

With the base shear calculated, we must now distribute the base shear on each level.

First, we must calculate the structural period exponent, $k$.

In [36]:
%%render
# fmt: off
if T <= 0.5: k = 1
elif T >= 2.5: k = 2
elif T > 0.5 and T < 2.5: k = 1 + ((T - 0.5) * (2 - 1)) / (2.5 - 0.5)
# fmt: on

<IPython.core.display.Latex object>

Now, we can distribute the forces per floor,

$$
F_x = C_{vx} V\;\;\text{(ASCE 7-16 Eq. 12.8-11)}
$$

As well as cumulatively from roof to ground.

$$
V_x = \sum_{i=x}^n F_i\;\; \text{(ASCE 7-16 Eq. 12.8-11)}
$$

In [ ]:
# Load in basic data
seismic_loads = pd.DataFrame(
    {
        "Level": ["Level 1", "Level 2", "Level 3", "Level 4", "Level 5", "Roof"],
        "floor height": [
            float(h_wall__1),
            float(h_wall__2),
            float(h_wall__3),
            float(h_wall__4),
            float(h_wall__5),
            0,
        ],
        "floor weight": [
            0,
            float(W_level__2),
            float(W_level__3),
            float(W_level__4),
            float(W_level__5),
            float(W_level__roof),
        ],
    },
)
seismic_loads = seismic_loads.set_index("Level")

# Calc cumulative elevation
elevations = [0]
for height in seismic_loads["floor height"][:-1]:
    elevations.append(height + elevations[-1])
seismic_loads.insert(2, "elevation", elevations)

# Calc coefficients and story forces ASCE 7-16 Eq. 12.8-11 and 12.8-12
seismic_loads["wxhxk"] = seismic_loads["floor weight"] * seismic_loads["elevation"] ** float(k)
cumulative_wxhxk = seismic_loads["wxhxk"].sum()
seismic_loads["C_vx"] = seismic_loads["wxhxk"] / cumulative_wxhxk
seismic_loads["F_x"] = seismic_loads["C_vx"] * float(V)

# Calc story shear forces ASCE 7-16 Eq. 12.8-13
story_shears = [0]
for shear in reversed(list(seismic_loads["F_x"][1:])):
    story_shears.insert(0, shear + story_shears[0])
seismic_loads["V_x"] = story_shears

# Calc overturning moments
floor_moments = seismic_loads["V_x"] * seismic_loads["floor height"]
overturning_moments = [0]
for floor_moment in reversed(list(floor_moments[:-1])):
    overturning_moments.insert(0, overturning_moments[0] + floor_moment)
seismic_loads["OTM"] = overturning_moments

In [38]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "floor height": "$h_{floor}$ [ft]",
    "floor weight": "$w_x$ [kip]",
    "elevation": "$h_x$ [ft]",
    "wxhxk": "$w_x h_x^k$",
    "C_vx": "$C_{vx}$",
    "F_x": "$F_x$ [kip]",
    "V_x": "$V_x$ [kip]",
    "OTM": "OTM [kip-ft]",
}

Calculating for each floor leads to the following \autoref{tab:seismic_forces_calculation_table}

In [39]:
# Format the dataframe
# Reverse the order of the dataframe with '.loc[::-1]' so roof is the top row and ground is the bottom row, then apply formatting with the 'format_dataframe' function
display_table(
    dataframe=seismic_loads.iloc[::-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Seismic Forces Calculation Table",
    label="tab:seismic_forces_calculation_table",
)

,$h_{floor}$ [ft],$w_x$ [kip],$h_x$ [ft],$w_x h_x^k$,$C_{vx}$,$F_x$ [kip],$V_x$ [kip],OTM [kip-ft]
Level,,,,,,,,
Roof,0.0,488.887917,53.0,25911.059583,0.249921,87.130707,0.000000,0.000000
Level 5,10.0,711.736583,43.0,30604.673083,0.295192,102.913847,87.130707,871.307073
Level 4,10.5,713.836083,32.5,23199.672708,0.223768,78.013170,190.044554,2866.774890
Level 3,10.5,713.836083,22.0,15704.393833,0.151474,52.808915,268.057724,5681.380990
Level 2,10.5,718.035083,11.5,8257.403458,0.079645,27.767039,320.866639,9050.480698
Ground,11.5,0.000000,0.0,0.000000,0.000000,0.000000,348.633678,13059.767994


# Diaphragm and Shear Wall Design

## Redundancy Factor

The redundancy factor, $\rho$, is either 1.3 or 1.0 according to ASCE 7-16 12.3.4.

No single story resists more than 35% of the shear force (see $C_{vx}$ in \autoref{tab:seismic_forces_calculation_table}), so all values of $\rho$ are $1.0$.

## Shear Wall Design

We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

In [ ]:
# Set up diaphragm dataframe
diaphragm_rigidity = ["Flexible", "Rigid"]
levels = ["Level 5", "Level 4", "Level 3", "Level 2", "Level 1"]
walls = {
    # N-S
    "E-2": "N-S",
    "E-2.4": "N-S",
    "E-3": "N-S",
    "E-3.3": "N-S",
    "E-3.4": "N-S",
    "E-4": "N-S",
    "E-4.8": "N-S",
    "E-5": "N-S",
    "E-5.9": "N-S",
    # "E-6": "N-S",
    "E-7.4": "N-S",
    "E-8--E-A.2": "N-S",
    "E-8--E-B": "N-S",
    "E-8--E-B.7": "N-S",
    "E-8--E-C.3": "N-S",
    "E-8--E-F": "N-S",
    "E-8--E-F.7": "N-S",
    "E-9--E-B.1": "N-S",
    "E-9--E-B.6": "N-S",
    "E-9--E-C.5": "N-S",
    "E-9--E-F.4": "N-S",
    # E-W
    "E-B": "E-W",
    "E-B.1": "E-W",
    "E-C": "E-W",
    "E-C.2": "E-W",
    "E-C.5": "E-W",
    "E-D": "E-W",
    "E-F.2": "E-W",
    "E-G--E-2": "E-W",
    "E-G--E-2.7": "E-W",
    "E-G--E-3.4": "E-W",
    "E-G--E-3.8": "E-W",
    "E-G--E-4.4": "E-W",
    # "E-G--E-5.7": "E-W",
    "E-G--E-7": "E-W",
    "E-H--E-1.2": "E-W",
    "E-H--E-2.4": "E-W",
    "E-H--E-3.1": "E-W",
    "E-H--E-4.1": "E-W",
    # "E-H--E-5.5": "E-W",
    "E-H--E-7": "E-W",
    "E-H--E-7.6": "E-W",
}

index = pd.MultiIndex.from_product(
    [levels, walls.keys()],
    names=["Level", "Wall"],
)

shear_walls = pd.DataFrame(index=index)
shear_walls["Direction"] = shear_walls.index.get_level_values("Wall").map(walls)
flexible = shear_walls.copy()
flexible_ns = flexible[flexible["Direction"] == "N-S"].copy()
flexible_ew = flexible[flexible["Direction"] == "E-W"].copy()
rigid = shear_walls.copy()
rigid_ns = rigid[rigid["Direction"] == "N-S"].copy()
rigid_ew = rigid[rigid["Direction"] == "E-W"].copy()

### Flexible Diaphragm Assumption

We will uniformly distribute the shear forces across the diaphragms based on tributary areas of each level. This removes some nuance in the distribution of floor weight distribution (corridors are 17% heavier than the rest of the floors, for instance), but the vast majority of the area of each level is typical floor weight (30 psf) instead of corridor weight (35 psf).

$$
f_{p,x} = F_x / A_{level}
$$

In [41]:
%%render
A_level
f_p__roof = seismic_loads.loc["Roof", "F_x"] * 1000 * lb / A_level
f_p__5 = seismic_loads.loc["Level 5", "F_x"] * 1000 * lb / A_level
f_p__4 = seismic_loads.loc["Level 4", "F_x"] * 1000 * lb / A_level
f_p__3 = seismic_loads.loc["Level 3", "F_x"] * 1000 * lb / A_level
f_p__2 = seismic_loads.loc["Level 2", "F_x"] * 1000 * lb / A_level

<IPython.core.display.Latex object>

#### North-South Direction

Most of the shear walls are not in line which presents an issue: if we assume a diaphragm is the full depth, there will be an extremely long collector element without a shear wall on both sides. We will avoid this by just assuming the flexible diaphragm does not extend the full depth and instead is just tributary to the nearest wall (\autoref{fig:diaphragm_trib_area}). This means that there are essentially two diaphragms through the depth of the building. This is a rough approximation to ensure all the shear load is resolved in the shear walls. \

Proper detailing will be done to ensure continuity through the depth of the diaphragm.

![Shear walls (thick red lines) in plan view with their (colored) tributary areas.](images/shear_wall_tributary_area.png){width=60% #fig:diaphragm_trib_area}

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ns} through \autoref{tab:shear_wall_forces_level2_ns}.

Shear wall sheathing will be assumed to have the following properties:

1. WSP Sheathing
2. 15/32" thickness
3. 10d common nails
4. Plywood panels
5. Framing uses Douglas-Fir-Larch (no specific gravity adjustment)

Shear capacity and stiffness values come from **AWC SDPWS 2021 Table 4.3A**.

Notes: 

- Cumulative shear force, $F_{x,cum}$, is cumulative from the Roof downwards. For example, Level 3 includes the shear forces from the Roof, Level 4, and Level 3.

In [42]:
from structural_tools.seismic.timber.sheathing import (
    DesignMethod,
    LoadCase,
    Nail,
    PanelType,
    Sheathing,
    SheathingMaterial,
    get_sheathing_properties,
)

# Set baseline sheathing properties. We set the number of sheathed sides later
sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
)

In [43]:
# Plan dimensions
l_E1__E2 = 26 + 1 / 12
l_E2__E3 = 24 + 8 / 12
l_E3__E4 = 26 + 4 / 12
l_E4__E5 = 21 + 4 / 12
l_E5__E6 = 24 + 8 / 12
l_E6__E7 = 8 + 9 / 12
l_E7__E8 = 30 + 9 / 12
l_E8__E9 = 5 + 6 / 12
l_E9__E10 = 30 + 9 / 12

l_E1__E24 = 35 + 11 / 12
l_E24__E34 = 24 + 8 / 12
l_E34__E48 = 34 + 4 / 12
l_E48__E59 = 24 + 8 / 12
l_E59__E74 = 24 + 8 / 12
l_E74__E9 = 23 + 10 / 12
l_E74__E92 = l_E74__E9 + float(Overlap)

l_E2__E24 = l_E1__E24 - l_E1__E2
l_E24__E3 = l_E2__E3 - l_E2__E24
l_E3__E33 = 8 + 10 / 12
l_E33__E4 = l_E3__E4 - l_E3__E33
l_E59__E6 = l_E1__E2 + l_E2__E3 + l_E3__E4 + l_E4__E5 + l_E5__E6 - (l_E1__E24 + l_E24__E34 + l_E34__E48 + l_E48__E59)
l_E59__E7 = (
    l_E1__E24
    + l_E24__E34
    + l_E34__E48
    + l_E48__E59
    + l_E59__E74
    - (l_E1__E2 + l_E2__E3 + l_E3__E4 + l_E4__E5 + l_E5__E6 + l_E6__E7)
)
l_E7__E74 = l_E7__E8 + l_E8__E9 - l_E74__E9

l_EA__EB = 35 + 11 / 12
l_EB__EC = 21 + 4 / 12
l_EC__ED = 17 + 11 / 12
l_ED__EF = 8 + 8 / 12
l_EF__EG = 30 + 9 / 12
l_EG__EH = 5 + 6 / 12
l_EH__EI = 30 + 9 / 12

l_EA__EB1 = 38 + 10.25 / 12
l_EB1__EC2 = 24 + 8.25 / 12
l_EC2__EF = 20 + 3.5 / 12
l_EF__EF2 = 6

l_EC2__EC5 = 8 + 10 / 12
l_EC5__EF = l_EC2__EF - l_EC2__EC5
l_EC2__EF2 = l_EC2__EF + l_EF__EF2
l_EC5__EF2 = l_EC5__EF + l_EF__EF2
l_EF2__EG = l_EF__EG - l_EF__EF2
l_EF2__EH = l_EF2__EG + l_EG__EH

d_trib_ns_1 = 67 / 2

In [44]:
# Wall tributary widths and lengths

# Zone 1 walls
t_w__E2 = l_E1__E2 + (l_E2__E3 / 2)
t_w__E3 = (l_E2__E3 / 2) + (l_E3__E33 / 2)
t_w__E33 = (l_E3__E33 / 2) + (l_E33__E4 / 2)
t_w__E4 = (l_E33__E4 / 2) + (l_E4__E5 / 2)
t_w__E5 = (l_E4__E5 / 2) + (l_E5__E6 / 2)
t_w__E6 = (l_E5__E6 / 2) + l_E6__E7

# Zone 2 walls
t_w__E24 = l_E1__E24 + (l_E24__E34 / 2) - float(L_jog)
t_w__E34 = (l_E24__E34 / 2) + (l_E34__E48 / 2)
t_w__E48 = (l_E34__E48 / 2) + (l_E48__E59 / 2)
t_w__E59 = (l_E48__E59 / 2) + l_E59__E7  # to reentrant corner
t_w__E74 = l_E7__E74 + l_E74__E92

# Zones 3 & 4 walls
t_w__E8 = l_E7__E8 + l_E8__E9 / 2
t_w__E9 = l_E8__E9 / 2 + l_E9__E10

# Zone 1 walls
t_l__E2 = d_trib_ns_1
t_l__E3 = d_trib_ns_1
t_l__E33 = d_trib_ns_1
t_l__E4 = d_trib_ns_1
t_l__E5 = d_trib_ns_1
t_l__E6 = d_trib_ns_1

# Zone 2 walls
t_l__E24 = d_trib_ns_1
t_l__E34 = d_trib_ns_1
t_l__E48 = d_trib_ns_1
t_l__E59 = d_trib_ns_1
t_l__E74 = l_EH__EI  # specific case for this bottom wall

# Zones 3 & 4 walls
t_l__E8__1 = 28
t_l__E8__2 = 18.33
t_l__E8__3 = 12.25 + 1 / 12  # make trib areas match
t_l__E8__4 = 17.75
t_l__E8__5 = 20
t_l__E8__6 = 18.75

t_l__E9__1 = 47.67
t_l__E9__2 = 20.67
t_l__E9__3 = 20.25 + 2 / 12  # make trib areas match
t_l__E9__4 = 31.5 + 0.5 / 12  # make trib areas match
# t_l__E8__1 + t_l__E8__2 + t_l__E8__3 + t_l__E8__4 + t_l__E8__5
# t_l__E9__1 + t_l__E9__2 + t_l__E9__3

In [45]:
# Wall segment lengths
l_zone__1 = l_EF__EG
l_zone__2 = l_EH__EI
l_stairwell = 18.75

l_E8__1 = 25
l_E8__2 = 12.5
l_E8__3 = 6
l_E8__4 = 11.5
l_E8__5 = 14.75
l_E8__6 = 7

l_E9__1 = 6
l_E9__2 = 12
l_E9__3 = 8.5
l_E9__4 = 24

In [ ]:
flexible_ns.loc["Level 5", "tributary weight"] = float(f_p__roof)
flexible_ns.loc["Level 4", "tributary weight"] = float(f_p__5)
flexible_ns.loc["Level 3", "tributary weight"] = float(f_p__4)
flexible_ns.loc["Level 2", "tributary weight"] = float(f_p__3)
flexible_ns.loc["Level 1", "tributary weight"] = float(f_p__2)

# Shear wall height is for the floor below
flexible_ns.loc["Level 5", "wall height"] = float(seismic_loads.loc["Level 5", "floor height"])
flexible_ns.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ns.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ns.loc["Level 2", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ns.loc["Level 1", "wall height"] = float(seismic_loads.loc["Level 1", "floor height"])

# Add wall tributary widths and wall lengths to df
wall_tributary = {
    # zone 1 walls
    "E-2": {"trib_width": float(t_w__E2), "trib_depth": t_l__E2, "wall_length": l_zone__1},
    "E-3": {"trib_width": float(t_w__E3), "trib_depth": t_l__E3, "wall_length": l_zone__1},
    "E-3.3": {"trib_width": float(t_w__E33), "trib_depth": t_l__E33, "wall_length": l_stairwell},
    "E-4": {"trib_width": float(t_w__E4), "trib_depth": t_l__E4, "wall_length": l_zone__1},
    "E-5": {"trib_width": float(t_w__E5), "trib_depth": t_l__E5, "wall_length": l_zone__1},
    # "E-6": {"trib_width": float(t_w__E6), "trib_depth": t_l__E6, "wall_length": l_zone__1},
    # zone 2 walls
    "E-2.4": {"trib_width": float(t_w__E24), "trib_depth": t_l__E24, "wall_length": l_zone__2},
    "E-3.4": {"trib_width": float(t_w__E34), "trib_depth": t_l__E34, "wall_length": l_zone__2},
    "E-4.8": {"trib_width": float(t_w__E48), "trib_depth": t_l__E48, "wall_length": l_zone__2},
    "E-5.9": {"trib_width": float(t_w__E59), "trib_depth": t_l__E59, "wall_length": l_zone__2},
    "E-7.4": {"trib_width": float(t_w__E74), "trib_depth": t_l__E74, "wall_length": l_zone__2},
    # zone 3 and 4 walls
    "E-8--E-A.2": {"trib_width": float(t_w__E8), "trib_depth": t_l__E8__1, "wall_length": l_E8__1},
    "E-8--E-B": {"trib_width": float(t_w__E8), "trib_depth": t_l__E8__2, "wall_length": l_E8__2},
    "E-8--E-B.7": {"trib_width": float(t_w__E8), "trib_depth": t_l__E8__3, "wall_length": l_E8__3},
    "E-8--E-C.3": {"trib_width": float(t_w__E8), "trib_depth": t_l__E8__4, "wall_length": l_E8__4},
    "E-8--E-F": {"trib_width": float(t_w__E8), "trib_depth": t_l__E8__5, "wall_length": l_E8__5},
    "E-8--E-F.7": {"trib_width": float(t_w__E8), "trib_depth": t_l__E8__6, "wall_length": l_E8__6},
    "E-9--E-B.1": {"trib_width": float(t_w__E9), "trib_depth": t_l__E9__1, "wall_length": l_E9__1},
    "E-9--E-B.6": {"trib_width": float(t_w__E9), "trib_depth": t_l__E9__2, "wall_length": l_E9__2},
    "E-9--E-C.5": {"trib_width": float(t_w__E9), "trib_depth": t_l__E9__3, "wall_length": l_E9__3},
    "E-9--E-F.4": {"trib_width": float(t_w__E9), "trib_depth": t_l__E9__4, "wall_length": l_E9__4},
}

flexible_ns["wall length"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary width"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary depth"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary area"] = flexible_ns["tributary depth"] * flexible_ns["tributary width"]

# Fix trib area of certain walls on all levels
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "E-5", "tributary area"] += 306
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "E-5.9", "tributary area"] += 400.2

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ns["tributary area"].sum() / len(flexible_ns.index.get_level_values("Level").unique())
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up. Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ns["line"] = flexible_ns.index.get_level_values("Wall").str.split("--").str[0]
# Get tributary area and wall length for the shear line
flexible_ns["line tributary area"] = flexible_ns.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ns["line wall length"] = flexible_ns.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ns["line shear demand"] = flexible_ns["line tributary area"] * flexible_ns["tributary weight"]
flexible_ns["line unit shear"] = flexible_ns["line shear demand"] / flexible_ns["line wall length"]
flexible_ns["line cumulative shear demand"] = flexible_ns.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ns["line cumulative unit shear"] = (
    flexible_ns["line cumulative shear demand"] / flexible_ns["line wall length"]
)

flexible_ns["shear demand"] = flexible_ns["line unit shear"] * flexible_ns["wall length"]
flexible_ns["cumulative shear demand"] = flexible_ns.groupby(level="Wall")["shear demand"].cumsum()
flexible_ns["unit shear demand"] = flexible_ns["cumulative shear demand"] / flexible_ns["wall length"]
flexible_ns["adjusted unit shear demand"] = flexible_ns["unit shear demand"] * LOAD_FACTOR_SEISMIC_ASD

flexible_ns["sheathed sides"] = 1
flexible_ns["aspect ratio"] = flexible_ns["wall height"] / flexible_ns["wall length"]

#### Check aspect ratios

We must check the aspect ratios of the shear walls to ensure we don't need to apply the aspect ratio factor in SDPWS Sec. 4.3.3.2. The maximum aspect ratio of all N-S walls is:

In [47]:
%%render
ratio_max = check_value(flexible_ns["aspect ratio"].max(), 2, "<")

<IPython.core.display.Latex object>

In [48]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

'$\\therefore$ All aspect ratios are OK'

In [49]:
flexible_ns[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(flexible_ns[["adjusted unit shear demand", "sheathed sides"]], sheathing=sheathing)
)

shear_dcr_check_value = 1

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ns["shear dcr"] > shear_dcr_check_value).any():
    flexible_ns.loc[flexible_ns["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ns.loc[
        flexible_ns["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ns.loc[
            flexible_ns["shear dcr"] > shear_dcr_check_value, ["adjusted unit shear demand", "sheathed sides"]
        ],
        sheathing=sheathing,
    )

In [50]:
# Update dataframe
flexible = flexible.reindex(columns=flexible_ns.columns)
flexible["line"] = flexible["line"].astype("string")
flexible.update(flexible_ns)

In [51]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$w_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [lbf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [52]:
def highlight_insufficient_capacity(row):
    styles = [""] * len(row)

    if row["$0.7 v$ [plf]"] > row["$v_{cap}$ [plf]"]:
        demand_idx = row.index.get_loc("$0.7 v$ [plf]")
        capacity_idx = row.index.get_loc("$v_{cap}$ [plf]")

        styles[demand_idx] = "background-color: red; font-weight: bold"
        styles[capacity_idx] = "background-color: red; font-weight: bold"

    return styles

In [ ]:
display_table(
    dataframe=flexible_ns,
    levels="Level 5",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 5",
    label="tab:shear_wall_forces_roof_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 4",
    label="tab:shear_wall_forces_level4_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 3",
    label="tab:shear_wall_forces_level3_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 2",
    label="tab:shear_wall_forces_level2_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 1",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 1",
    label="tab:shear_wall_forces_level1_ns",
)

,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
E-2,4.874195,1286.958333,6272.886138,6272.886138,30.75,203.996297,142.797408,310.714286,1,6.0
E-2.4,4.874195,1448.875000,7062.099579,7062.099579,30.75,229.661775,160.763242,310.714286,1,6.0
E-3,4.874195,561.125000,2735.032785,2735.032785,30.75,88.944156,62.260909,310.714286,1,6.0
E-3.3,4.874195,441.083333,2149.926269,2149.926269,18.75,114.662734,80.263914,310.714286,1,6.0
E-3.4,4.874195,988.250000,4816.923412,4816.923412,30.75,156.647916,109.653541,310.714286,1,6.0
E-4,4.874195,650.458333,3170.460890,3170.460890,30.75,103.104419,72.173093,310.714286,1,6.0
E-4.8,4.874195,988.250000,4816.923412,4816.923412,30.75,156.647916,109.653541,310.714286,1,6.0
E-5,4.874195,1076.500000,5247.071139,5247.071139,30.75,170.636460,119.445522,310.714286,1,6.0
E-5.9,4.874195,1229.325000,5991.970021,5991.970021,30.75,194.860814,136.402570,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
E-2,5.757123,1286.958333,7409.177110,13682.063248,30.75,444.945146,311.461602,460.714286,1,4.0
E-2.4,5.757123,1448.875000,8341.351236,15403.450815,30.75,500.925230,350.647661,460.714286,1,4.0
E-3,5.757123,561.125000,3230.465508,5965.498293,30.75,193.999944,135.799961,310.714286,1,6.0
E-3.3,5.757123,441.083333,2539.370897,4689.297165,18.75,250.095849,175.067094,310.714286,1,6.0
E-3.4,5.757123,988.250000,5689.476566,10506.399978,30.75,341.671544,239.170081,310.714286,1,6.0
E-4,5.757123,650.458333,3744.768474,6915.229364,30.75,224.885508,157.419855,310.714286,1,6.0
E-4.8,5.757123,988.250000,5689.476566,10506.399978,30.75,341.671544,239.170081,310.714286,1,6.0
E-5,5.757123,1076.500000,6197.542649,11444.613788,30.75,372.182562,260.527794,310.714286,1,6.0
E-5.9,5.757123,1229.325000,7077.374935,13069.344956,30.75,425.019348,297.513544,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
E-2,4.364149,1286.958333,5616.478357,19298.541605,30.75,627.594849,439.316394,460.714286,1,4.0
E-2.4,4.364149,1448.875000,6323.106870,21726.557685,30.75,706.554721,494.588305,600.000000,1,3.0
E-3,4.364149,561.125000,2448.833296,8414.331589,30.75,273.636800,191.545760,310.714286,1,6.0
E-3.3,4.364149,441.083333,1924.953537,6614.250702,18.75,352.760037,246.932026,310.714286,1,6.0
E-3.4,4.364149,988.250000,4312.870582,14819.270560,30.75,481.927498,337.349249,460.714286,1,4.0
E-4,4.364149,650.458333,2838.697304,9753.926668,30.75,317.200867,222.040607,310.714286,1,6.0
E-4.8,4.364149,988.250000,4312.870582,14819.270560,30.75,481.927498,337.349249,460.714286,1,4.0
E-5,4.364149,1076.500000,4698.006761,16142.620549,30.75,524.963270,367.474289,460.714286,1,4.0
E-5.9,4.364149,1229.325000,5364.957883,18434.302839,30.75,599.489523,419.642666,460.714286,1,4.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
E-2,2.954193,1286.958333,3801.923811,23100.465415,30.75,751.234648,525.864253,600.000000,1,3.0
E-2.4,2.954193,1448.875000,4280.256958,26006.814643,30.75,845.750070,592.025049,600.000000,1,3.0
E-3,2.954193,561.125000,1657.671770,10072.003359,30.75,327.544825,229.281377,310.714286,1,6.0
E-3.3,2.954193,441.083333,1303.045471,7917.296173,18.75,422.255796,295.579057,310.714286,1,6.0
E-3.4,2.954193,988.250000,2919.481625,17738.752184,30.75,576.869990,403.808993,460.714286,1,4.0
E-4,2.954193,650.458333,1921.579713,11675.506381,30.75,379.691264,265.783885,310.714286,1,6.0
E-4.8,2.954193,988.250000,2919.481625,17738.752184,30.75,576.869990,403.808993,460.714286,1,4.0
E-5,2.954193,1076.500000,3180.189192,19322.809741,30.75,628.384057,439.868840,460.714286,1,4.0
E-5.9,2.954193,1229.325000,3631.663798,22065.966637,30.75,717.592411,502.314688,600.000000,1,3.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
E-2,1.553321,1286.958333,1999.059572,25099.524988,30.75,816.244715,571.371301,600.000000,1,3.0
E-2.4,1.553321,1448.875000,2250.568152,28257.382795,30.75,918.939278,643.257495,769.642857,1,2.0
E-3,1.553321,561.125000,871.607319,10943.610678,30.75,355.889778,249.122845,310.714286,1,6.0
E-3.3,1.553321,441.083333,685.144062,8602.440234,18.75,458.796813,321.157769,460.714286,1,4.0
E-3.4,1.553321,988.250000,1535.069607,19273.821791,30.75,626.790953,438.753667,460.714286,1,4.0
E-4,1.553321,650.458333,1010.370673,12685.877055,30.75,412.548847,288.784193,310.714286,1,6.0
E-4.8,1.553321,988.250000,1535.069607,19273.821791,30.75,626.790953,438.753667,460.714286,1,4.0
E-5,1.553321,1076.500000,1672.150196,20994.959937,30.75,682.762925,477.934047,600.000000,1,3.0
E-5.9,1.553321,1229.325000,1909.536498,23975.503135,30.75,779.691159,545.783811,600.000000,1,3.0


#### East-West Direction

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ew} through \autoref{tab:shear_wall_forces_level2_ew}.

In [54]:
# Plan dimensions
d_trib_ew_1 = 67 / 2

In [55]:
# Wall tributary widths and lengths

# zone 3 walls
t_w__EB = l_EA__EB - float(L_jog) + (l_EB__EC / 2)
t_w__EC = (l_EB__EC / 2) + (l_EC__ED / 2)
t_w__ED = (l_EC__ED / 2) + l_ED__EF

# zone 4 walls
t_w__EB1 = l_EA__EB1 + (l_EB1__EC2 / 2)
t_w__EC2 = (l_EB1__EC2 / 2) + (l_EC2__EC5 / 2)
t_w__EC5 = (l_EC2__EC5 / 2) + (l_EC5__EF2 / 2)
t_w__EF2 = (l_EC5__EF2 / 2) + l_EF2__EH

# zone 1 and 2 walls
t_w__G = l_EF__EG + l_EG__EH / 2
t_w__H = l_EG__EH / 2 + l_EH__EI

# zone 3 walls
t_l__EB = d_trib_ew_1
t_l__EC = d_trib_ew_1
t_l__ED = d_trib_ew_1

# zone 4 walls
t_l__EB1 = d_trib_ew_1
t_l__EC2 = d_trib_ew_1
t_l__EC5 = d_trib_ew_1
t_l__EF2 = d_trib_ew_1

# zone 1 and 2 walls
t_l__EG__1 = 40.75
t_l__EG__2 = 14
t_l__EG__3 = 16
t_l__EG__4 = 12.5
t_l__EG__5 = 24.75
t_l__EG__6 = 21.25
t_l__EG__7 = 36
t_l__EH__1 = 27.33
t_l__EH__2 = 18.66
t_l__EH__3 = 25.66
t_l__EH__4 = 33
t_l__EH__5 = 18.66
t_l__EH__6 = 18.75
t_l__EH__7 = 27.5

In [56]:
# Wall segment lengths
l_zone__3 = l_E7__E8
l_zone__4 = l_E9__E10
l_stairwell = 18.75

l_EG__1 = 11.75
l_EG__2 = 6.75
l_EG__3 = 7.75
l_EG__4 = 6
l_EG__5 = 12.25
l_EG__6 = 10.5
l_EG__7 = 7.5

l_EH__1 = 23.25
l_EH__2 = 12.33
l_EH__3 = 20
l_EH__4 = 27
l_EH__5 = 12
l_EH__6 = 13
l_EH__7 = 23.33

In [ ]:
flexible_ew.loc["Level 5", "tributary weight"] = float(f_p__roof)
flexible_ew.loc["Level 4", "tributary weight"] = float(f_p__5)
flexible_ew.loc["Level 3", "tributary weight"] = float(f_p__4)
flexible_ew.loc["Level 2", "tributary weight"] = float(f_p__3)
flexible_ew.loc["Level 1", "tributary weight"] = float(f_p__2)

# Shear wall height is for the floor below
flexible_ew.loc["Level 5", "wall height"] = float(seismic_loads.loc["Level 5", "floor height"])
flexible_ew.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ew.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ew.loc["Level 2", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ew.loc["Level 1", "wall height"] = float(seismic_loads.loc["Level 1", "floor height"])

# Add wall tributary widths and wall lengths to df
wall_tributary = {
    # E-W
    # zone 3 walls
    "E-B": {"trib_width": float(t_w__EB), "trib_depth": t_l__EB, "wall_length": l_zone__3},
    "E-C": {"trib_width": float(t_w__EC), "trib_depth": t_l__EC, "wall_length": l_zone__3},
    "E-D": {"trib_width": float(t_w__ED), "trib_depth": t_l__ED, "wall_length": l_zone__3},
    # zone 4 walls
    "E-B.1": {"trib_width": float(t_w__EB1), "trib_depth": t_l__EB1, "wall_length": l_zone__4},
    "E-C.2": {"trib_width": float(t_w__EC2), "trib_depth": t_l__EC2, "wall_length": l_stairwell},
    "E-C.5": {"trib_width": float(t_w__EC5), "trib_depth": t_l__EC5, "wall_length": l_stairwell},
    "E-F.2": {"trib_width": float(t_w__EF2), "trib_depth": t_l__EF2, "wall_length": l_zone__4},
    # zones 1 and 2 walls
    "E-G--E-2": {"trib_width": float(t_w__G), "trib_depth": t_l__EG__1, "wall_length": l_EG__1},
    "E-G--E-2.7": {"trib_width": float(t_w__G), "trib_depth": t_l__EG__2, "wall_length": l_EG__2},
    "E-G--E-3.4": {"trib_width": float(t_w__G), "trib_depth": t_l__EG__3, "wall_length": l_EG__3},
    "E-G--E-3.8": {"trib_width": float(t_w__G), "trib_depth": t_l__EG__4, "wall_length": l_EG__4},
    "E-G--E-4.4": {"trib_width": float(t_w__G), "trib_depth": t_l__EG__5, "wall_length": l_EG__5},
    # "E-G--E-5.7": {"trib_width": float(t_w__G), "trib_depth": t_l__EG__6, "wall_length": l_EG__6},
    "E-G--E-7": {"trib_width": float(t_w__G), "trib_depth": t_l__EG__7, "wall_length": l_EG__7},
    "E-H--E-1.2": {"trib_width": float(t_w__H), "trib_depth": t_l__EH__1, "wall_length": l_EH__1},
    "E-H--E-2.4": {"trib_width": float(t_w__H), "trib_depth": t_l__EH__2, "wall_length": l_EH__2},
    "E-H--E-3.1": {"trib_width": float(t_w__H), "trib_depth": t_l__EH__3, "wall_length": l_EH__3},
    "E-H--E-4.1": {"trib_width": float(t_w__H), "trib_depth": t_l__EH__4, "wall_length": l_EH__4},
    # "E-H--E-5.5": {"trib_width": float(t_w__H), "trib_depth": t_l__EH__5, "wall_length": l_EH__5},
    "E-H--E-7": {"trib_width": float(t_w__H), "trib_depth": t_l__EH__6, "wall_length": l_EH__6},
    "E-H--E-7.6": {"trib_width": float(t_w__H), "trib_depth": t_l__EH__7, "wall_length": l_EH__7},
}

flexible_ew["wall length"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary width"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary depth"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary area"] = flexible_ew["tributary depth"] * flexible_ew["tributary width"]

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ns["tributary area"].sum() / len(flexible_ns.index.get_level_values("Level").unique())
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up.\n\
    Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ew["line"] = flexible_ew.index.get_level_values("Wall").str.split("--").str[0]

# Get tributary area and wall length for the shear line
flexible_ew["line tributary area"] = flexible_ew.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ew["line wall length"] = flexible_ew.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ew["line shear demand"] = flexible_ew["line tributary area"] * flexible_ew["tributary weight"]
flexible_ew["line unit shear"] = flexible_ew["line shear demand"] / flexible_ew["line wall length"]
flexible_ew["line cumulative shear demand"] = flexible_ew.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ew["line cumulative unit shear"] = (
    flexible_ew["line cumulative shear demand"] / flexible_ew["line wall length"]
)

flexible_ew["shear demand"] = flexible_ew["line unit shear"] * flexible_ew["wall length"]
flexible_ew["cumulative shear demand"] = flexible_ew.groupby(level="Wall")["shear demand"].cumsum()
flexible_ew["unit shear demand"] = flexible_ew["cumulative shear demand"] / flexible_ew["wall length"]
flexible_ew["adjusted unit shear demand"] = flexible_ew["unit shear demand"] * LOAD_FACTOR_SEISMIC_ASD

flexible_ew["sheathed sides"] = 1
flexible_ew["aspect ratio"] = flexible_ew["wall height"] / flexible_ew["wall length"]
flexible_ew["aspect ratio factor"] = np.where(
    flexible_ew["aspect ratio"] > 2, 1.25 - 0.125 * flexible_ew["wall height"] / flexible_ew["wall length"], 1
)

#### Check aspect ratios

We will also check the aspect ratios of the E-W walls.

In [58]:
%%render
ratio_max = check_value(flexible_ew["aspect ratio"].max(), 2, "<=")

<IPython.core.display.Latex object>

In [59]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

'$\\therefore$ All aspect ratios are OK'

In [60]:
# using sheathing from the NS calc in memory

flexible_ew[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        flexible_ew[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ew["shear dcr"] > shear_dcr_check_value).any():
    flexible_ew.loc[flexible_ew["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ew.loc[
        flexible_ew["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ew.loc[
            flexible_ew["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

In [61]:
# Update dataframe
flexible.update(flexible_ew)

In [62]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$W_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [ ]:
display_table(
    dataframe=flexible_ew,
    levels="Level 5",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 5",
    label="tab:shear_wall_forces_roof_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 3",
    label="tab:shear_wall_forces_level3_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 2",
    label="tab:shear_wall_forces_level2_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 1",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 1",
    label="tab:shear_wall_forces_level1_ew",
)

,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
E-B,4.874195,1393.041667,6789.957013,6789.957013,30.75,220.811610,154.568127,310.714286,1,6.0
E-B.1,4.874195,1715.130208,8359.879439,8359.879439,30.75,271.865998,190.306199,310.714286,1,6.0
E-C,4.874195,657.437500,3204.478711,3204.478711,30.75,104.210690,72.947483,310.714286,1,6.0
E-C.2,4.874195,561.473958,2736.733676,2736.733676,18.75,145.959129,102.171391,310.714286,1,6.0
E-C.5,4.874195,440.385417,2146.524487,2146.524487,18.75,114.481306,80.136914,310.714286,1,6.0
E-D,4.874195,590.437500,2877.907632,2877.907632,30.75,93.590492,65.513344,310.714286,1,6.0
E-F.2,4.874195,1305.802083,6364.734254,6364.734254,30.75,206.983228,144.888259,310.714286,1,6.0
E-G--E-2,4.874195,1365.125000,5313.060243,5313.060243,11.75,452.175340,316.522738,460.714286,1,4.0
E-G--E-2.7,4.874195,469.000000,3052.183544,3052.183544,6.75,452.175340,316.522738,460.714286,1,4.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
E-B,4.364149,1393.041667,6079.441865,20889.310761,30.75,679.327179,475.529025,600.000000,1,3.0
E-B.1,4.364149,1715.130208,7485.084361,25719.178955,30.75,836.396064,585.477245,600.000000,1,3.0
E-C,4.364149,657.437500,2869.155429,9858.582533,30.75,320.604310,224.423017,310.714286,1,6.0
E-C.2,4.364149,561.473958,2450.356203,8419.564382,18.75,449.043434,314.330404,460.714286,1,4.0
E-C.5,4.364149,440.385417,1921.907724,6603.785115,18.75,352.201873,246.541311,310.714286,1,6.0
E-D,4.364149,590.437500,2576.757424,8853.886224,30.75,287.931259,201.551882,310.714286,1,6.0
E-F.2,4.364149,1305.802083,5698.715296,19581.112442,30.75,636.784144,445.748901,460.714286,1,4.0
E-G--E-2,4.364149,1365.125000,4757.090629,16345.636106,11.75,1391.117966,973.782577,1200.000000,2,3.0
E-G--E-2.7,4.364149,469.000000,2732.796744,9390.046274,6.75,1391.117966,973.782577,1200.000000,2,3.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
E-B,2.954193,1393.041667,4115.314494,25004.625254,30.75,813.158545,569.210981,600.000000,1,3.0
E-B.1,2.954193,1715.130208,5066.826337,30786.005292,30.75,1001.170904,700.819633,769.642857,1,2.0
E-C,2.954193,657.437500,1942.197521,11800.780055,30.75,383.765205,268.635644,310.714286,1,6.0
E-C.2,2.954193,561.473958,1658.702660,10078.267043,18.75,537.507576,376.255303,460.714286,1,4.0
E-C.5,2.954193,440.385417,1300.983690,7904.768805,18.75,421.587670,295.111369,310.714286,1,6.0
E-D,2.954193,590.437500,1744.266564,10598.152788,30.75,344.655375,241.258763,310.714286,1,6.0
E-F.2,2.954193,1305.802083,3857.591892,23438.704334,30.75,762.234287,533.564001,600.000000,1,3.0
E-G--E-2,2.954193,1365.125000,3220.184426,19565.820532,11.75,1665.176215,1165.623351,1200.000000,2,3.0
E-G--E-2.7,2.954193,469.000000,1849.893181,11239.939455,6.75,1665.176215,1165.623351,1200.000000,2,3.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
E-B,1.553321,1393.041667,2163.841056,27168.466310,30.75,883.527360,618.469152,769.642857,1,2.0
E-B.1,1.553321,1715.130208,2664.147993,33450.153285,30.75,1087.809863,761.466904,769.642857,1,2.0
E-C,1.553321,657.437500,1021.211560,12821.991615,30.75,416.975337,291.882736,310.714286,1,6.0
E-C.2,1.553321,561.473958,872.149363,10950.416406,18.75,584.022208,408.815546,460.714286,1,4.0
E-C.5,1.553321,440.385417,684.059973,8588.828778,18.75,458.070868,320.649608,460.714286,1,4.0
E-D,1.553321,590.437500,917.139045,11515.291833,30.75,374.481035,262.136725,310.714286,1,6.0
E-F.2,1.553321,1305.802083,2028.329968,25467.034302,30.75,828.196237,579.737366,600.000000,1,3.0
E-G--E-2,1.553321,1365.125000,1693.179775,21259.000307,11.75,1809.276622,1266.493635,1539.285714,2,2.0
E-G--E-2.7,1.553321,469.000000,972.677743,12212.617197,6.75,1809.276622,1266.493635,1539.285714,2,2.0


### Rigid Diaphragm Assumption

With a rigid diaphragm assumption, we now must calculate the following in each direction:

1. The relative stiffnesses of the shear walls to obtain a center of rigidity.
2. The center of mass of the diaphragm with the center of rigidity to obtain an eccentricity.

The relative stiffnesses of the shear walls can be calculated by using the deflection equation from NDS SDPWS Sec. 4.3.4

\begin{align*}
k &= \frac{F}{\delta},\, \text{assume }F = \text{induced load demand in the shear wall} \\
\delta_{sw} &= \frac{8vh^3}{EAb} + \frac{vh}{1000 G_a} + \frac{h \Delta_A}{b}\quad \text{(SDPWS Eq. 4.3-1)}
\end{align*}

where

- $b =$ the shear wall length [ft]
- $\Delta_A =$ vertical deformation of the wall overturning anchorage system + vertical compression deformation [in]
- $E =$ modulus of elasticity of end posts [psi]
- $A =$ area of end post cross-section [in$^2$]
- $G_a =$ apparent shear wall shear stiffness from nail slip and panel shear deformation [kip/in]
- $h =$ shear wall height [ft]
- $v =$ unit shear force induced by the design load [plf]
- $\delta_{sw} =$ maximum shear wall deflection determined by elastic analysis [in]

Note that for the following calculations, we will conservatively use the unit shear force induced by the nominal load, not the design load, for the value of $v$. This is because the stiffnesses obtained from this use will allow us to use the stiffnesses for deflection limits and story drift calculations

We will assume that the end posts are (2) 2x4 Douglas Fir No. 2; this means:

In [64]:
%%render
E_endpost = 1_600_000 * psi
A_endpost = 2 * 5.25 * inch**2

<IPython.core.display.Latex object>

We will assume a continuous tiedown rod system. A conservative first-pass value for $\Delta_A$ can be:

In [65]:
%%render params
Delta_A = 0.25 * inch

<IPython.core.display.Latex object>

#### Shear Wall Rigidities

With the deflections of each shear wall, along with the force demand of each shear wall, we can obtain the stiffnesses (and therefore, relative stiffnesses) of all the walls.

Each relative stiffness is relative to the level. The sum of all relative stiffnesses on each level is $1.0$.

\begin{align*}
k &= \frac{F_x}{\delta_{sw}} \\
R &= \frac{k_{wall}}{\sum {k_{wall}}}
\end{align*}

##### North-South Direction

Rigidities in the N-S direction can be found in \autoref{tab:wall_rigidities_ns_roof} through \autoref{tab:wall_rigidities_ns_level2}.

In [ ]:
rigid_ns["unit shear demand"] = flexible_ns["unit shear demand"]
rigid_ns["wall length"] = flexible_ns["wall length"]
rigid_ns["wall height"] = flexible_ns["wall height"]
rigid_ns["shear stiffness"] = flexible_ns["shear stiffness"]
rigid_ns["sheathed sides"] = flexible_ns["sheathed sides"]
rigid_ns["nail spacing"] = flexible_ns["nail spacing"]
rigid_ns["end post youngs modulus"] = float(E_endpost)
rigid_ns["end post area"] = float(A_endpost)
rigid_ns["Delta_A"] = float(Delta_A)

rigid_ns["delta_sw"] = (
    8
    * rigid_ns["unit shear demand"]
    * rigid_ns["wall height"] ** 3
    / (rigid_ns["end post youngs modulus"] * rigid_ns["end post area"] * rigid_ns["wall length"])
    + rigid_ns["unit shear demand"]
    * rigid_ns["wall height"]
    / (1000 * rigid_ns["shear stiffness"] * rigid_ns["sheathed sides"])
    + rigid_ns["wall height"] * rigid_ns["Delta_A"] / rigid_ns["wall length"]
)

In [67]:
rigid_ns["cumulative shear demand"] = flexible_ns["cumulative shear demand"]
rigid_ns["wall stiffness"] = rigid_ns["cumulative shear demand"] / rigid_ns["delta_sw"]
rigid_ns["total level stiffness"] = rigid_ns.groupby("Level")["wall stiffness"].transform("sum")
rigid_ns["total level cumulative shear"] = rigid_ns.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ns["relative wall stiffness"] = rigid_ns["wall stiffness"] / rigid_ns["total level stiffness"]
rigid_ns["direct shear force"] = rigid_ns["total level cumulative shear"] * rigid_ns["relative wall stiffness"]

In [68]:
column_name_map = {
    "cumulative shear demand": "$F_x$ [lbf]",
    "delta_sw": r"$\delta_{sw}$ [in]",
    "wall stiffness": "$k$ [lbf/in]",
    "relative wall stiffness": "$R$ [-]",
}

In [69]:
display_table(
    dataframe=rigid_ns,
    levels="Level 5",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 5",
    label="tab:wall_rigidities_ns_level5",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 4",
    label="tab:wall_rigidities_ns_level4",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 3",
    label="tab:wall_rigidities_ns_level3",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 2",
    label="tab:wall_rigidities_ns_level2",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 1",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 1",
    label="tab:wall_rigidities_ns_level1",
)

,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
E-2,6272.886138,0.230172,27253.095026,0.083047
E-2.4,7062.099579,0.248901,28373.075266,0.086460
E-3,2735.032785,0.146210,18706.229603,0.057002
E-3.3,2149.926269,0.218147,9855.385412,0.030032
E-3.4,4816.923412,0.195618,24624.130455,0.075036
E-4,3170.460890,0.156543,20252.908155,0.061715
E-4.8,4816.923412,0.195618,24624.130455,0.075036
E-5,5247.071139,0.205826,25492.695793,0.077682
E-5.9,5991.970021,0.223505,26809.145787,0.081694


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
E-2,13682.063248,0.368161,37163.223394,0.084466
E-2.4,15403.450815,0.403741,38151.826428,0.086713
E-3,5965.498293,0.234344,25456.201441,0.057858
E-3.3,4689.297165,0.334925,14001.048897,0.031822
E-3.4,10506.399978,0.347745,30212.978161,0.068670
E-4,6915.229364,0.258061,26796.830235,0.060905
E-4.8,10506.399978,0.347745,30212.978161,0.068670
E-5,11444.613788,0.371175,30833.485752,0.070080
E-5.9,13069.344956,0.411750,31741.001107,0.072143


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
E-2,19298.541605,0.484249,39852.537621,0.078964
E-2.4,21726.557685,0.488497,44476.375254,0.088125
E-3,8414.331589,0.295499,28475.002011,0.056420
E-3.3,6614.250702,0.414941,15940.213046,0.031584
E-3.4,14819.270560,0.391666,37836.462463,0.074969
E-4,9753.926668,0.328953,29651.437561,0.058751
E-4.8,14819.270560,0.391666,37836.462463,0.074969
E-5,16142.620549,0.419019,38524.812111,0.076333
E-5.9,18434.302839,0.466386,39525.871454,0.078316


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
E-2,23100.465415,0.513989,44943.492939,0.081036
E-2.4,26006.814643,0.567916,45793.444602,0.082569
E-3,10072.003359,0.336896,29896.448820,0.053906
E-3.3,7917.296173,0.469106,16877.407982,0.030431
E-3.4,17738.752184,0.452009,39244.214640,0.070760
E-4,11675.506381,0.376941,30974.363479,0.055849
E-4.8,17738.752184,0.452009,39244.214640,0.070760
E-5,19322.809741,0.484750,39861.363012,0.071873
E-5.9,22065.966637,0.494794,44596.248236,0.080410


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
E-2,25099.524988,0.606763,41366.271890,0.078941
E-2.4,28257.382795,0.574608,49176.757620,0.093846
E-3,10943.610678,0.394216,27760.449146,0.052976
E-3.3,8602.440234,0.481417,17868.996301,0.034100
E-3.4,19273.821791,0.532264,36211.032252,0.069103
E-4,12685.877055,0.442092,28695.124416,0.054760
E-4.8,19273.821791,0.532264,36211.032252,0.069103
E-5,20994.959937,0.522828,40156.559313,0.076632
E-5.9,23975.503135,0.583778,41069.583242,0.078375


##### East-West Direction

Rigidities in the E-W direction can be found in \autoref{tab:wall_rigidities_ew_roof}, \autoref{tab:wall_rigidities_ew_level4}, \autoref{tab:wall_rigidities_ew_level3}, \autoref{tab:wall_rigidities_ew_level2},

In [ ]:
rigid_ew["unit shear demand"] = flexible_ew["unit shear demand"]
rigid_ew["wall length"] = flexible_ew["wall length"]
rigid_ew["wall height"] = flexible_ew["wall height"]
rigid_ew["shear stiffness"] = flexible_ew["shear stiffness"]
rigid_ew["sheathed sides"] = flexible_ew["sheathed sides"]
rigid_ew["nail spacing"] = flexible_ew["nail spacing"]
rigid_ew["end post youngs modulus"] = float(E_endpost)
rigid_ew["end post area"] = float(A_endpost)
rigid_ew["Delta_A"] = float(Delta_A)

rigid_ew["delta_sw"] = (
    8
    * rigid_ew["unit shear demand"]
    * rigid_ew["wall height"] ** 3
    / (rigid_ew["end post youngs modulus"] * rigid_ew["end post area"] * rigid_ew["wall length"])
    + rigid_ew["unit shear demand"]
    * rigid_ew["wall height"]
    / (1000 * rigid_ew["shear stiffness"] * rigid_ew["sheathed sides"])
    + rigid_ew["wall height"] * rigid_ew["Delta_A"] / rigid_ew["wall length"]
)

In [71]:
rigid_ew["cumulative shear demand"] = flexible_ew["cumulative shear demand"]
rigid_ew["wall stiffness"] = rigid_ew["cumulative shear demand"] / rigid_ew["delta_sw"]
rigid_ew["total level stiffness"] = rigid_ew.groupby("Level")["wall stiffness"].transform("sum")
rigid_ew["total level cumulative shear"] = rigid_ew.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ew["relative wall stiffness"] = rigid_ew["wall stiffness"] / rigid_ew["total level stiffness"]
rigid_ew["direct shear force"] = rigid_ew["total level cumulative shear"] * rigid_ew["relative wall stiffness"]

In [72]:
column_name_map = {
    "cumulative shear demand": "$F_x$ [lbf]",
    "delta_sw": r"$\delta_{sw}$ [in]",
    "wall stiffness": "$k$ [lbf/in]",
    "relative wall stiffness": "$R$ [-]",
}

In [73]:
display_table(
    dataframe=rigid_ew,
    levels="Level 5",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 5",
    label="tab:wall_rigidities_ew_level5",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 4",
    label="tab:wall_rigidities_ns_level4",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 3",
    label="tab:wall_rigidities_ew_level3",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 2",
    label="tab:wall_rigidities_ew_level2",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 1",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 1",
    label="tab:wall_rigidities_ew_level1",
)

,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
E-B,6789.957013,0.242443,28006.422848,0.100955
E-B.1,8359.879439,0.279701,29888.640517,0.107739
E-C,3204.478711,0.157351,20365.186856,0.073410
E-C.2,2736.733676,0.241297,11341.775816,0.040884
E-C.5,2146.524487,0.218013,9845.848395,0.035491
E-D,2877.907632,0.149600,19237.286877,0.069344
E-F.2,6364.734254,0.232351,27392.721727,0.098742
E-G--E-2,5313.060243,0.497077,10688.612675,0.038529
E-G--E-2.7,3052.183544,0.668255,4567.391045,0.016464


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
E-2,13682.063248,0.368161,37163.223394,0.084466
E-2.4,15403.450815,0.403741,38151.826428,0.086713
E-3,5965.498293,0.234344,25456.201441,0.057858
E-3.3,4689.297165,0.334925,14001.048897,0.031822
E-3.4,10506.399978,0.347745,30212.978161,0.068670
E-4,6915.229364,0.258061,26796.830235,0.060905
E-4.8,10506.399978,0.347745,30212.978161,0.068670
E-5,11444.613788,0.371175,30833.485752,0.070080
E-5.9,13069.344956,0.411750,31741.001107,0.072143


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
E-B,20889.310761,0.472962,44167.025162,0.103412
E-B.1,25719.178955,0.562579,45716.591774,0.107040
E-C,9858.582533,0.331567,29733.348778,0.069617
E-C.2,8419.564382,0.430552,19555.268178,0.045786
E-C.5,6603.785115,0.414506,15931.694330,0.037302
E-D,8853.886224,0.306476,28889.330439,0.067641
E-F.2,19581.112442,0.490089,39954.178370,0.093548
E-G--E-2,16345.636106,1.057444,15457.683029,0.036192
E-G--E-2.7,9390.046274,1.271273,7386.335914,0.017294


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
E-B,25004.625254,0.549320,45519.208586,0.097131
E-B.1,30786.005292,0.560370,54938.715003,0.117230
E-C,11800.780055,0.380069,31049.009679,0.066254
E-C.2,10078.267043,0.487793,20660.963412,0.044087
E-C.5,7904.768805,0.468585,16869.429360,0.035997
E-D,10598.152788,0.350036,30277.325487,0.064607
E-F.2,23438.704334,0.520265,45051.469431,0.096132
E-G--E-2,19565.820532,1.221755,16014.523395,0.034172
E-G--E-2.7,11239.939455,1.445107,7777.927572,0.016597


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
E-B,27168.466310,0.556069,48858.127408,0.108696
E-B.1,33450.153285,0.663021,50451.118707,0.112240
E-C,12821.991615,0.445832,28759.691871,0.063982
E-C.2,10950.416406,0.570965,19178.778443,0.042668
E-C.5,8588.828778,0.480898,17859.981280,0.039734
E-D,11515.291833,0.409925,28091.205262,0.062495
E-F.2,25467.034302,0.614278,41458.460391,0.092234
E-G--E-2,21259.000307,1.260836,16861.032265,0.037511
E-G--E-2.7,12212.617197,1.524687,8009.920016,0.017820


#### Center of Rigidity and Center of Mass

We can find the center of rigidity by taking the weighted sum of all wall stiffnesses and their locations relative to a datum. In this case, we will use the north-west-most corner of the building (i.e., the intersection of lines B-1 and B-A).

$$
\bar{x}_r = \frac{\sum k_{ns} x}{\sum k_{ns}},\, \bar{y}_r = \frac{\sum k_{ew} y}{\sum k_{ew}}
$$

Note: each wall is assumed to have zero rigidity in the direction perpendicular to them.

We will get the center of mass, $CM_i$, for each direction by taking the weighted sum of the centroids of each of the zones 1, 2, 3, and 4.

In [74]:
set_params_columns(4)

In [75]:
%%render params
x_1 = (131 + 10 / 12) / 2
x_2 = ((170 + 2 / 12) / 2) + float(L_jog)
x_3 = ((30 + (9 / 12) + 5 + (6 / 12)) / 2) + (131 + 10 / 12)
x_4 = ((30 + (9 / 12)) / 2) + (131 + (10 / 12) + 30 + (9 / 12) + 5 + (6 / 12))

y_1 = ((30 + (9 / 12) + 5 + (6 / 12)) / 2) + (83 + 10 / 12)
y_2 = ((30 + (9 / 12)) / 2) + (83 + (10 / 12) + 30 + (9 / 12) + 5 + (6 / 12))
y_3 = (115 + 1 / 12) / 2 + float(L_jog)
y_4 = (120 + 1 / 12) / 2

A_1 = L_1 * w_1
A_2 = L_2 * w_2
A_3 = L_3 * w_3
A_4 = L_4 * w_4

<IPython.core.display.Latex object>

In [76]:
%%render long
CM_x = (x_1 * A_1 + x_2 * A_2 + x_3 * A_3 + x_4 * A_4) / A_level
CM_y = (y_1 * A_1 + y_2 * A_2 + y_3 * A_3 + y_4 * A_4) / A_level

<IPython.core.display.Latex object>

An accidental torsion eccentricity of 5% must be added/subtracted.

In [77]:
%%render
X_plan = 198 + 10 / 12
Y_plan = 150 + 10 / 12
CM_x__a__plus = CM_x + X_plan * 0.05
CM_x__a__minus = CM_x - X_plan * 0.05
CM_y__a__plus = CM_y + Y_plan * 0.05
CM_y__a__minus = CM_y - Y_plan * 0.05

<IPython.core.display.Latex object>

The eccentricity used is determined by the location of the wall in relation to the center of rigidity. Since the wall rotates about the center of rigidity, we want to position the center of mass to ensure each wall is getting the maximum possible force to be conservative. If the center of mass is positioned on the opposite side of the center of rigidity relative to the wall in question, that mass will _decrease_ the shear experienced by that wall.

This means we always want the center of mass to be on the same side of the center of rigidity as the wall. The eccentricities shown in the following tables are based on the worst-case scenario. This is not immediately obvious due to all torques being given a positive value. 

How the center of rigidity, center of mass, and eccentricity interact are illustrated in \autoref{fig:rigid_diaphragm_moments}. The moments generated by the rotation of the center of mass around the center of rigidity differ based on the chosen eccentricity.

![Moments applied by a rigid diaphragm](images/rigid_diaphragm_moments.png){width=50% #fig:rigid_diaphragm_moments}

$$
d_x = \left| x - \text{CR}_x \right|\text{ and }\, d_y = \left| y - \text{CR}_y \right|
$$

In [78]:
# Update dataframe
rigid = rigid.reindex(columns=rigid_ns.columns)
rigid.update(rigid_ns)
rigid.update(rigid_ew)

In [79]:
# Location of wall lines of rigidity
wall_rigidity_loc = {
    # N-S
    # zone 1 walls
    "E-2": {"x": float(l_E1__E2), "y": 0},
    "E-3": {"x": float(l_E1__E2 + l_E2__E3), "y": 0},
    "E-3.3": {"x": float(l_E1__E2 + l_E2__E3 + l_E3__E33), "y": 0},
    "E-4": {"x": float(l_E1__E2 + l_E2__E3 + l_E3__E4), "y": 0},
    "E-5": {"x": float(l_E1__E2 + l_E2__E3 + l_E3__E4 + l_E4__E5), "y": 0},
    # "E-6": {"x": float(l_E1__E2 + l_E2__E3 + l_E3__E4 + l_E4__E5 + l_E5__E6), "y": 0},
    # zone 2 walls
    "E-2.4": {"x": float(l_E1__E24), "y": 0},
    "E-3.4": {"x": float(l_E1__E24 + l_E24__E34), "y": 0},
    "E-4.8": {"x": float(l_E1__E24 + l_E24__E34 + l_E34__E48), "y": 0},
    "E-5.9": {"x": float(l_E1__E24 + l_E24__E34 + l_E34__E48 + l_E48__E59), "y": 0},
    "E-7.4": {"x": float(l_E1__E24 + l_E24__E34 + l_E34__E48 + l_E48__E59 + l_E59__E74), "y": 0},
    # zone 3 and 4 walls
    "E-8--E-A.2": {"x": float(l_E1__E2 + l_E2__E3 + l_E3__E4 + l_E4__E5 + l_E5__E6 + l_E6__E7 + l_E7__E8), "y": 0},
    "E-8--E-B": {"x": float(l_E1__E2 + l_E2__E3 + l_E3__E4 + l_E4__E5 + l_E5__E6 + l_E6__E7 + l_E7__E8), "y": 0},
    "E-8--E-B.7": {"x": float(l_E1__E2 + l_E2__E3 + l_E3__E4 + l_E4__E5 + l_E5__E6 + l_E6__E7 + l_E7__E8), "y": 0},
    "E-8--E-C.3": {"x": float(l_E1__E2 + l_E2__E3 + l_E3__E4 + l_E4__E5 + l_E5__E6 + l_E6__E7 + l_E7__E8), "y": 0},
    "E-8--E-F": {"x": float(l_E1__E2 + l_E2__E3 + l_E3__E4 + l_E4__E5 + l_E5__E6 + l_E6__E7 + l_E7__E8), "y": 0},
    "E-8--E-F.7": {"x": float(l_E1__E2 + l_E2__E3 + l_E3__E4 + l_E4__E5 + l_E5__E6 + l_E6__E7 + l_E7__E8), "y": 0},
    "E-9--E-B.1": {
        "x": float(l_E1__E2 + l_E2__E3 + l_E3__E4 + l_E4__E5 + l_E5__E6 + l_E6__E7 + l_E7__E8 + l_E8__E9),
        "y": 0,
    },
    "E-9--E-B.6": {
        "x": float(l_E1__E2 + l_E2__E3 + l_E3__E4 + l_E4__E5 + l_E5__E6 + l_E6__E7 + l_E7__E8 + l_E8__E9),
        "y": 0,
    },
    "E-9--E-C.5": {
        "x": float(l_E1__E2 + l_E2__E3 + l_E3__E4 + l_E4__E5 + l_E5__E6 + l_E6__E7 + l_E7__E8 + l_E8__E9),
        "y": 0,
    },
    "E-9--E-F.4": {
        "x": float(l_E1__E2 + l_E2__E3 + l_E3__E4 + l_E4__E5 + l_E5__E6 + l_E6__E7 + l_E7__E8 + l_E8__E9),
        "y": 0,
    },
    # E-W
    # zone 3 walls
    "E-B": {"x": 0, "y": float(l_EA__EB)},
    "E-C": {"x": 0, "y": float(l_EA__EB + l_EB__EC)},
    "E-D": {"x": 0, "y": float(l_EA__EB + l_EB__EC + l_EC__ED)},
    # zone 4 walls
    "E-B.1": {"x": 0, "y": float(l_EA__EB1)},
    "E-C.2": {"x": 0, "y": float(l_EA__EB1 + l_EB1__EC2)},
    "E-C.5": {"x": 0, "y": float(l_EA__EB1 + l_EB1__EC2 + l_EC2__EC5)},
    "E-F.2": {"x": 0, "y": float(l_EA__EB1 + l_EB1__EC2 + l_EC2__EC5 + l_EC5__EF2)},
    # zones 1 and 2 walls
    "E-G--E-2": {"x": 0, "y": float(l_EA__EB + l_EB__EC + l_EC__ED + l_ED__EF + l_EF__EG)},
    "E-G--E-2.7": {"x": 0, "y": float(l_EA__EB + l_EB__EC + l_EC__ED + l_ED__EF + l_EF__EG)},
    "E-G--E-3.4": {"x": 0, "y": float(l_EA__EB + l_EB__EC + l_EC__ED + l_ED__EF + l_EF__EG)},
    "E-G--E-3.8": {"x": 0, "y": float(l_EA__EB + l_EB__EC + l_EC__ED + l_ED__EF + l_EF__EG)},
    "E-G--E-4.4": {"x": 0, "y": float(l_EA__EB + l_EB__EC + l_EC__ED + l_ED__EF + l_EF__EG)},
    # "E-G--E-5.7": {"x": 0, "y": float(l_EA__EB + l_EB__EC + l_EC__ED + l_ED__EF + l_EF__EG)},
    "E-G--E-7": {"x": 0, "y": float(l_EA__EB + l_EB__EC + l_EC__ED + l_ED__EF + l_EF__EG)},
    "E-H--E-1.2": {"x": 0, "y": float(l_EA__EB + l_EB__EC + l_EC__ED + l_ED__EF + l_EF__EG + l_EG__EH)},
    "E-H--E-2.4": {"x": 0, "y": float(l_EA__EB + l_EB__EC + l_EC__ED + l_ED__EF + l_EF__EG + l_EG__EH)},
    "E-H--E-3.1": {"x": 0, "y": float(l_EA__EB + l_EB__EC + l_EC__ED + l_ED__EF + l_EF__EG + l_EG__EH)},
    "E-H--E-4.1": {"x": 0, "y": float(l_EA__EB + l_EB__EC + l_EC__ED + l_ED__EF + l_EF__EG + l_EG__EH)},
    # "E-H--E-5.5": {"x": 0, "y": float(l_EA__EB + l_EB__EC + l_EC__ED + l_ED__EF + l_EF__EG + l_EG__EH)},
    "E-H--E-7": {"x": 0, "y": float(l_EA__EB + l_EB__EC + l_EC__ED + l_ED__EF + l_EF__EG + l_EG__EH)},
    "E-H--E-7.6": {"x": 0, "y": float(l_EA__EB + l_EB__EC + l_EC__ED + l_ED__EF + l_EF__EG + l_EG__EH)},
    # N-S
}

In [80]:
# Calc centers of rigidity
rigid["x"] = rigid.index.get_level_values("Wall").map({key: value["x"] for key, value in wall_rigidity_loc.items()})
rigid["y"] = rigid.index.get_level_values("Wall").map({key: value["y"] for key, value in wall_rigidity_loc.items()})

rigid["kx"] = rigid["relative wall stiffness"] * rigid["x"]
rigid["CRx"] = rigid.groupby(["Level", "Direction"])["kx"].transform("sum")  # all relative wall rigidities sum to 1
rigid["ky"] = rigid["relative wall stiffness"] * rigid["y"]
rigid["CRy"] = rigid.groupby(["Level", "Direction"])["ky"].transform("sum")  # all relative wall rigidities sum to 1

# x eccentricity. Use the "+" eccentricity for walls on the + side of the CR. "-" for - side
CMx = CM_x
CMxa_plus = CM_x__a__plus
CMxa_minus = CM_x__a__minus
rigid["ex"] = np.where(rigid["x"] <= rigid["CRx"], rigid["CRx"] - CMxa_minus, CMxa_plus - rigid["CRx"])
rigid["ex"] = np.where(rigid["ex"] <= 0, 0, rigid["ex"])  # assign 0s to negatives

# y eccentricity. Use the "+" eccentricity for walls on the + side of the CR. "-" for - side
CMy = CM_y
CMya_plus = CM_y__a__plus
CMya_minus = CM_y__a__minus
rigid["ey"] = np.where(rigid["y"] <= rigid["CRy"], rigid["CRy"] - CMya_minus, CMya_plus - rigid["CRy"])
rigid["ey"] = np.where(rigid["ey"] <= 0, 0, rigid["ey"])  # assign 0s to negatives

# distance from CR
rigid["dx"] = abs(rigid["x"] - rigid["CRx"])
rigid["dy"] = abs(rigid["y"] - rigid["CRy"])

# Torque
rigid["torque x"] = rigid["total level cumulative shear"] * rigid["ex"]
rigid["torque y"] = rigid["total level cumulative shear"] * rigid["ey"]

# Polar moment of inertia
rigid["Jx wall"] = rigid["relative wall stiffness"] * rigid["dx"] ** 2
rigid["Jy wall"] = rigid["relative wall stiffness"] * rigid["dy"] ** 2
rigid["J"] = (rigid["Jx wall"] + rigid["Jy wall"]).groupby(["Level"]).transform("sum")

rigid["torsional shear force x"] = rigid["torque x"] * rigid["relative wall stiffness"] * rigid["dx"] / rigid["J"]
rigid["torsional shear force y"] = rigid["torque y"] * rigid["relative wall stiffness"] * rigid["dy"] / rigid["J"]
rigid["torsional shear force"] = np.where(
    rigid["torsional shear force x"] == 0, rigid["torsional shear force y"], rigid["torsional shear force x"]
)
rigid["shear force demand"] = rigid["torsional shear force"] + rigid["direct shear force"]

The centers of rigidity for each floor are given in \autoref{tab:center_of_rigidity_roof} through \autoref{tab:center_of_rigidity_level2}.

In [81]:
column_name_map = {
    "x": "$x$ [ft]",
    "y": "$y$ [ft]",
    "CRx": "CR$_x$ [ft]",
    "dx": "$d_x$ [ft]",
    "ex": "e$_x$ [ft]",
    "CRy": "CR$_y$ [ft]",
    "dy": "$d_y$ [ft]",
    "ey": "e$_y$ [ft]",
}
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 5",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 5",
    label="tab:center_of_rigidity_level5",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 4",
    label="tab:center_of_rigidity_level4",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 3",
    label="tab:center_of_rigidity_level3",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 2",
    label="tab:center_of_rigidity_level2",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 1",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 1",
    label="tab:center_of_rigidity_level1",
)

,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
E-2,26.083333,0.000000,103.492478,77.409144,0.000000,0.000000,0.000000,0.000000
E-2.4,35.916667,0.000000,103.492478,67.575811,0.000000,0.000000,0.000000,0.000000
E-3,50.750000,0.000000,103.492478,52.742478,0.000000,0.000000,0.000000,0.000000
E-3.3,59.583333,0.000000,103.492478,43.909144,0.000000,0.000000,0.000000,0.000000
E-3.4,60.583333,0.000000,103.492478,42.909144,0.000000,0.000000,0.000000,0.000000
E-4,77.083333,0.000000,103.492478,26.409144,0.000000,0.000000,0.000000,0.000000
E-4.8,94.916667,0.000000,103.492478,8.575811,0.000000,0.000000,0.000000,0.000000
E-5,98.416667,0.000000,103.492478,5.075811,0.000000,0.000000,0.000000,0.000000
E-5.9,119.583333,0.000000,103.492478,16.090856,23.333179,0.000000,0.000000,0.000000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
E-2,26.083333,0.000000,105.081636,78.998303,0.000000,0.00000,0.000000,0.000000
E-2.4,35.916667,0.000000,105.081636,69.164970,0.000000,0.00000,0.000000,0.000000
E-3,50.750000,0.000000,105.081636,54.331636,0.000000,0.00000,0.000000,0.000000
E-3.3,59.583333,0.000000,105.081636,45.498303,0.000000,0.00000,0.000000,0.000000
E-3.4,60.583333,0.000000,105.081636,44.498303,0.000000,0.00000,0.000000,0.000000
E-4,77.083333,0.000000,105.081636,27.998303,0.000000,0.00000,0.000000,0.000000
E-4.8,94.916667,0.000000,105.081636,10.164970,0.000000,0.00000,0.000000,0.000000
E-5,98.416667,0.000000,105.081636,6.664970,0.000000,0.00000,0.000000,0.000000
E-5.9,119.583333,0.000000,105.081636,14.501697,21.744021,0.00000,0.000000,0.000000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
E-2,26.083333,0.000000,104.288371,78.205038,0.000000,0.000000,0.000000,0.000000
E-2.4,35.916667,0.000000,104.288371,68.371704,0.000000,0.000000,0.000000,0.000000
E-3,50.750000,0.000000,104.288371,53.538371,0.000000,0.000000,0.000000,0.000000
E-3.3,59.583333,0.000000,104.288371,44.705038,0.000000,0.000000,0.000000,0.000000
E-3.4,60.583333,0.000000,104.288371,43.705038,0.000000,0.000000,0.000000,0.000000
E-4,77.083333,0.000000,104.288371,27.205038,0.000000,0.000000,0.000000,0.000000
E-4.8,94.916667,0.000000,104.288371,9.371704,0.000000,0.000000,0.000000,0.000000
E-5,98.416667,0.000000,104.288371,5.871704,0.000000,0.000000,0.000000,0.000000
E-5.9,119.583333,0.000000,104.288371,15.294962,22.537286,0.000000,0.000000,0.000000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
E-2,26.083333,0.000000,106.245982,80.162649,0.000000,0.000000,0.000000,0.000000
E-2.4,35.916667,0.000000,106.245982,70.329316,0.000000,0.000000,0.000000,0.000000
E-3,50.750000,0.000000,106.245982,55.495982,0.000000,0.000000,0.000000,0.000000
E-3.3,59.583333,0.000000,106.245982,46.662649,0.000000,0.000000,0.000000,0.000000
E-3.4,60.583333,0.000000,106.245982,45.662649,0.000000,0.000000,0.000000,0.000000
E-4,77.083333,0.000000,106.245982,29.162649,0.000000,0.000000,0.000000,0.000000
E-4.8,94.916667,0.000000,106.245982,11.329316,0.000000,0.000000,0.000000,0.000000
E-5,98.416667,0.000000,106.245982,7.829316,0.000000,0.000000,0.000000,0.000000
E-5.9,119.583333,0.000000,106.245982,13.337351,20.579675,0.000000,0.000000,0.000000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
E-2,26.083333,0.000000,105.004254,78.920921,0.000000,0.000000,0.000000,0.000000
E-2.4,35.916667,0.000000,105.004254,69.087587,0.000000,0.000000,0.000000,0.000000
E-3,50.750000,0.000000,105.004254,54.254254,0.000000,0.000000,0.000000,0.000000
E-3.3,59.583333,0.000000,105.004254,45.420921,0.000000,0.000000,0.000000,0.000000
E-3.4,60.583333,0.000000,105.004254,44.420921,0.000000,0.000000,0.000000,0.000000
E-4,77.083333,0.000000,105.004254,27.920921,0.000000,0.000000,0.000000,0.000000
E-4.8,94.916667,0.000000,105.004254,10.087587,0.000000,0.000000,0.000000,0.000000
E-5,98.416667,0.000000,105.004254,6.587587,0.000000,0.000000,0.000000,0.000000
E-5.9,119.583333,0.000000,105.004254,14.579079,21.821403,0.000000,0.000000,0.000000


And now, we can get the forces in the shear walls using the rigid diaphragm assumption in \autoref{tab:rigid_forces_roof} through \autoref{tab:rigid_forces_level2}. These forces can be compared to the flexible shear wall forces obtained earlier, $F_{flex}$.

In [82]:
rigid["flexible shear force demand"] = flexible["cumulative shear demand"]

In [83]:
column_name_map = {
    "torque x": "$T_x$ [lbf-ft]",
    "torque y": "$T_y$ [lbf-ft]",
    "dx": "$d_x$ [ft]",
    "dy": "$d_y$ [ft]",
    "relative wall stiffness": "$R$ [-]",
    "torsional shear force": "$F_t$ [lbf]",
    "direct shear force": "$F_v$ [lbf]",
    "shear force demand": "$F_{tot}$ [lbf]",
    "flexible shear force demand": "$F_{flex}$ [lbf]",
}
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 5",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 5",
    label="tab:rigid_forces_level5",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 4",
    label="tab:rigid_forces_level4",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 3",
    label="tab:rigid_forces_level3",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 2",
    label="tab:rigid_forces_level2",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 1",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 1",
    label="tab:rigid_forces_level1",
)

,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
E-2,0.000000e+00,0.000000e+00,77.409144,0.000000,0.083047,0.000000,7235.913411,7235.913411,6272.886138
E-2.4,0.000000e+00,0.000000e+00,67.575811,0.000000,0.086460,0.000000,7533.277070,7533.277070,7062.099579
E-3,0.000000e+00,0.000000e+00,52.742478,0.000000,0.057002,0.000000,4966.652688,4966.652688,2735.032785
E-3.3,0.000000e+00,0.000000e+00,43.909144,0.000000,0.030032,0.000000,2616.683184,2616.683184,2149.926269
E-3.4,0.000000e+00,0.000000e+00,42.909144,0.000000,0.075036,0.000000,6537.902415,6537.902415,4816.923412
E-4,0.000000e+00,0.000000e+00,26.409144,0.000000,0.061715,0.000000,5377.308141,5377.308141,3170.460890
E-4.8,0.000000e+00,0.000000e+00,8.575811,0.000000,0.075036,0.000000,6537.902415,6537.902415,4816.923412
E-5,0.000000e+00,0.000000e+00,5.075811,0.000000,0.077682,0.000000,6768.513419,6768.513419,5247.071139
E-5.9,2.033036e+06,0.000000e+00,16.090856,0.000000,0.081694,734.278561,7118.041359,7852.319920,5991.970021


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
E-2,0.000000e+00,0.000000e+00,78.998303,0.000000,0.084466,0.000000,16052.388793,16052.388793,13682.063248
E-2.4,0.000000e+00,0.000000e+00,69.164970,0.000000,0.086713,0.000000,16479.408810,16479.408810,15403.450815
E-3,0.000000e+00,0.000000e+00,54.331636,0.000000,0.057858,0.000000,10995.624314,10995.624314,5965.498293
E-3.3,0.000000e+00,0.000000e+00,45.498303,0.000000,0.031822,0.000000,6047.653026,6047.653026,4689.297165
E-3.4,0.000000e+00,0.000000e+00,44.498303,0.000000,0.068670,0.000000,13050.280028,13050.280028,10506.399978
E-4,0.000000e+00,0.000000e+00,27.998303,0.000000,0.060905,0.000000,11574.699342,11574.699342,6915.229364
E-4.8,0.000000e+00,0.000000e+00,10.164970,0.000000,0.068670,0.000000,13050.280028,13050.280028,10506.399978
E-5,0.000000e+00,0.000000e+00,6.664970,0.000000,0.070080,0.000000,13318.303849,13318.303849,11444.613788
E-5.9,4.132333e+06,0.000000e+00,14.501697,0.000000,0.072143,1161.505624,13710.298621,14871.804245,13069.344956


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
E-2,0.000000e+00,0.000000e+00,78.205038,0.000000,0.078964,0.000000,21166.830835,21166.830835,19298.541605
E-2.4,0.000000e+00,0.000000e+00,68.371704,0.000000,0.088125,0.000000,23622.684209,23622.684209,21726.557685
E-3,0.000000e+00,0.000000e+00,53.538371,0.000000,0.056420,0.000000,15123.893899,15123.893899,8414.331589
E-3.3,0.000000e+00,0.000000e+00,44.705038,0.000000,0.031584,0.000000,8466.306367,8466.306367,6614.250702
E-3.4,0.000000e+00,0.000000e+00,43.705038,0.000000,0.074969,0.000000,20096.035238,20096.035238,14819.270560
E-4,0.000000e+00,0.000000e+00,27.205038,0.000000,0.058751,0.000000,15748.732711,15748.732711,9753.926668
E-4.8,0.000000e+00,0.000000e+00,9.371704,0.000000,0.074969,0.000000,20096.035238,20096.035238,14819.270560
E-5,0.000000e+00,0.000000e+00,5.871704,0.000000,0.076333,0.000000,20461.637567,20461.637567,16142.620549
E-5.9,6.041293e+06,0.000000e+00,15.294962,0.000000,0.078316,1974.951062,20993.329023,22968.280085,18434.302839


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
E-2,0.000000e+00,0.000000e+00,80.162649,0.000000,0.081036,0.000000,26001.887148,26001.887148,23100.465415
E-2.4,0.000000e+00,0.000000e+00,70.329316,0.000000,0.082569,0.000000,26493.623455,26493.623455,26006.814643
E-3,0.000000e+00,0.000000e+00,55.495982,0.000000,0.053906,0.000000,17296.476921,17296.476921,10072.003359
E-3.3,0.000000e+00,0.000000e+00,46.662649,0.000000,0.030431,0.000000,9764.360289,9764.360289,7917.296173
E-3.4,0.000000e+00,0.000000e+00,45.662649,0.000000,0.070760,0.000000,22704.591334,22704.591334,17738.752184
E-4,0.000000e+00,0.000000e+00,29.162649,0.000000,0.055849,0.000000,17920.100353,17920.100353,11675.506381
E-4.8,0.000000e+00,0.000000e+00,11.329316,0.000000,0.070760,0.000000,22704.591334,22704.591334,17738.752184
E-5,0.000000e+00,0.000000e+00,7.829316,0.000000,0.071873,0.000000,23061.640181,23061.640181,19322.809741
E-5.9,6.603331e+06,0.000000e+00,13.337351,0.000000,0.080410,1911.685813,25800.990044,27712.675857,22065.966637


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
E-2,0.000000e+00,0.000000e+00,78.920921,0.000000,0.078941,0.000000,27521.424934,27521.424934,25099.524988
E-2.4,0.000000e+00,0.000000e+00,69.087587,0.000000,0.093846,0.000000,32717.824970,32717.824970,28257.382795
E-3,0.000000e+00,0.000000e+00,54.254254,0.000000,0.052976,0.000000,18469.324945,18469.324945,10943.610678
E-3.3,0.000000e+00,0.000000e+00,45.420921,0.000000,0.034100,0.000000,11888.435140,11888.435140,8602.440234
E-3.4,0.000000e+00,0.000000e+00,44.420921,0.000000,0.069103,0.000000,24091.588639,24091.588639,19273.821791
E-4,0.000000e+00,0.000000e+00,27.920921,0.000000,0.054760,0.000000,19091.174440,19091.174440,12685.877055
E-4.8,0.000000e+00,0.000000e+00,10.087587,0.000000,0.069103,0.000000,24091.588639,24091.588639,19273.821791
E-5,0.000000e+00,0.000000e+00,6.587587,0.000000,0.076632,0.000000,26716.590165,26716.590165,20994.959937
E-5.9,7.607676e+06,0.000000e+00,14.579079,0.000000,0.078375,2332.895305,27324.034791,29656.930096,23975.503135


### Final Shear Wall Nailing Selection

With the envelope procedure complete, we can now select the shear wall sheathing and nailing specifications in \autoref{tab:shear_walls_roof} through \autoref{tab:shear_walls_level2}.

In [ ]:
shear_walls["wall length"] = rigid["wall length"]
shear_walls["rigid shear force demand"] = rigid["shear force demand"]
shear_walls["flexible shear force demand"] = rigid["flexible shear force demand"]
shear_walls["shear force demand"] = np.where(
    shear_walls["flexible shear force demand"] >= shear_walls["rigid shear force demand"],
    shear_walls["flexible shear force demand"],
    shear_walls["rigid shear force demand"],
)
shear_walls["unit shear demand"] = shear_walls["shear force demand"] / shear_walls["wall length"]
shear_walls["adjusted unit shear demand"] = (
    shear_walls["shear force demand"] / shear_walls["wall length"] * LOAD_FACTOR_SEISMIC_ASD
)
shear_walls["sheathed sides"] = 1

# using sheathing from the NS calc in memory

shear_walls[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        shear_walls[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (shear_walls["shear dcr"] > shear_dcr_check_value).any():
    shear_walls.loc[shear_walls["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    shear_walls.loc[
        shear_walls["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        shear_walls.loc[
            shear_walls["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

column_name_map = {
    "shear force demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 5",
    position_float="centering",
    caption="Final shear wall design values - Level 5",
    label="tab:shear_walls_level5",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    caption="Final shear wall design values - Level 4",
    label="tab:shear_walls_level4",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    caption="Final shear wall design values - Level 3",
    label="tab:shear_walls_level3",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    caption="Final shear wall design values - Level 2",
    label="tab:shear_walls_level2",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 1",
    position_float="centering",
    caption="Final shear wall design values - Level 1",
    label="tab:shear_walls_level1",
)

,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
E-2,7235.913411,30.75,235.314257,164.719980,310.714286,1,6.0
E-2.4,7533.277070,30.75,244.984620,171.489234,310.714286,1,6.0
E-3,4966.652688,30.75,161.517161,113.062012,310.714286,1,6.0
E-3.3,2616.683184,18.75,139.556436,97.689506,310.714286,1,6.0
E-3.4,6537.902415,30.75,212.614713,148.830299,310.714286,1,6.0
E-4,5377.308141,30.75,174.871809,122.410267,310.714286,1,6.0
E-4.8,6537.902415,30.75,212.614713,148.830299,310.714286,1,6.0
E-5,6768.513419,30.75,220.114258,154.079980,310.714286,1,6.0
E-5.9,7852.319920,30.75,255.359997,178.751998,310.714286,1,6.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
E-2,16052.388793,30.75,522.028904,365.420233,460.714286,1,4.0
E-2.4,16479.408810,30.75,535.915734,375.141014,460.714286,1,4.0
E-3,10995.624314,30.75,357.581278,250.306895,310.714286,1,6.0
E-3.3,6047.653026,18.75,322.541495,225.779046,310.714286,1,6.0
E-3.4,13050.280028,30.75,424.399351,297.079545,310.714286,1,6.0
E-4,11574.699342,30.75,376.412987,263.489091,310.714286,1,6.0
E-4.8,13050.280028,30.75,424.399351,297.079545,310.714286,1,6.0
E-5,13318.303849,30.75,433.115572,303.180901,310.714286,1,6.0
E-5.9,14871.804245,30.75,483.635910,338.545137,460.714286,1,4.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
E-2,21166.830835,30.75,688.352222,481.846556,600.000000,1,3.0
E-2.4,23622.684209,30.75,768.217373,537.752161,600.000000,1,3.0
E-3,15123.893899,30.75,491.833948,344.283764,460.714286,1,4.0
E-3.3,8466.306367,18.75,451.536340,316.075438,460.714286,1,4.0
E-3.4,20096.035238,30.75,653.529601,457.470721,460.714286,1,4.0
E-4,15748.732711,30.75,512.153909,358.507737,460.714286,1,4.0
E-4.8,20096.035238,30.75,653.529601,457.470721,460.714286,1,4.0
E-5,20461.637567,30.75,665.419108,465.793375,600.000000,1,3.0
E-5.9,22968.280085,30.75,746.935938,522.855156,600.000000,1,3.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
E-2,26001.887148,30.75,845.589826,591.912878,600.000000,1,3.0
E-2.4,26493.623455,30.75,861.581251,603.106875,769.642857,1,2.0
E-3,17296.476921,30.75,562.487054,393.740938,460.714286,1,4.0
E-3.3,9764.360289,18.75,520.765882,364.536117,460.714286,1,4.0
E-3.4,22704.591334,30.75,738.360694,516.852486,600.000000,1,3.0
E-4,17920.100353,30.75,582.767491,407.937244,460.714286,1,4.0
E-4.8,22704.591334,30.75,738.360694,516.852486,600.000000,1,3.0
E-5,23061.640181,30.75,749.972038,524.980427,600.000000,1,3.0
E-5.9,27712.675857,30.75,901.225231,630.857662,769.642857,1,2.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
E-2,27521.424934,30.75,895.005689,626.503982,769.642857,1,2.0
E-2.4,32717.824970,30.75,1063.994308,744.796016,769.642857,1,2.0
E-3,18469.324945,30.75,600.628454,420.439917,460.714286,1,4.0
E-3.3,11888.435140,18.75,634.049874,443.834912,460.714286,1,4.0
E-3.4,24091.588639,30.75,783.466297,548.426408,600.000000,1,3.0
E-4,19091.174440,30.75,620.851201,434.595841,460.714286,1,4.0
E-4.8,24091.588639,30.75,783.466297,548.426408,600.000000,1,3.0
E-5,26716.590165,30.75,868.832201,608.182540,769.642857,1,2.0
E-5.9,29656.930096,30.75,964.453011,675.117108,769.642857,1,2.0


### Shear Wall Axial Forces

#### Tension Forces

The shear forces get converted to overturning forces in the end posts of the shear walls. This overturning moment is counteracted by the dead load tributary to each shear wall according to ASCE 7-16 Sec. 2.4.5. The applicable equation is:

$$
0.6 D - 0.7 E_v + 0.7 E_h \quad\text{(ASCE 7-16 Eq. 2.4.5-10)}
$$

There are no vertical seismic effects, $E_v$, so we will only consider the horizontal seismic effects, $E_h$, and dead loads, $D$.

The overturning moment is:

$$
M_{OT} = V h_{wall}
$$

Using the area dead loads from earlier, along with the tributary areas for each shear wall, we can get the dead load that is applied to each wall. This resists the overturning moment at the centroid of the wall.

$$
M_R = \frac{D l_{wall}}{2}
$$

We will assume that the holddowns are 6" inset from the end posts. The tension caused by horizontal load effects counteracted by dead loads is:

$$
T_{seismic} = \frac{M_{OT} - M_R}{l_{wall} - 0.5}\quad\text{(units in feet)}
$$

In [85]:
shear_walls["line"] = flexible["line"]
shear_walls["wall height"] = rigid["wall height"]
shear_walls["tributary area"] = flexible["tributary area"]
shear_walls["floor OTM"] = shear_walls["shear force demand"] * shear_walls["wall height"]
shear_walls["OTM"] = shear_walls.groupby(level="Wall")["floor OTM"].cumsum()
shear_walls["dead load"] = 0.6 * float(U_floor * 1000) * shear_walls["tributary area"]

# Reduce resisting moment dead load in some walls that definitely are not correct
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "E-2", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "E-2.4", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "E-3.4", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "E-5", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "E-7.4", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls["line"] == "E-8", "dead load"] *= 0
shear_walls.loc[shear_walls["line"] == "E-9", "dead load"] *= 0
shear_walls.loc[shear_walls["line"] == "E-B.1", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls["line"] == "E-D", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls["line"] == "E-F.2", "dead load"] *= 1 / 16
shear_walls.loc[shear_walls["line"] == "E-G", "dead load"] *= 0
shear_walls.loc[shear_walls["line"] == "E-H", "dead load"] *= 0

shear_walls["RM"] = shear_walls["dead load"] * shear_walls["wall length"] / 2

shear_walls["total moment"] = shear_walls["OTM"] - shear_walls["RM"]
shear_walls["total moment"] = np.where(
    shear_walls["RM"] <= shear_walls["OTM"], shear_walls["OTM"] - shear_walls["RM"], 0
)
wall_arm_shortening_length = 0.5  # feet due to holddown positioning
shear_walls["tension force"] = shear_walls["total moment"] / (shear_walls["wall length"] - wall_arm_shortening_length)
num_bins = 5
shear_walls["binned tension force"] = (
    shear_walls["tension force"]
    .groupby(level="Level")
    .transform(lambda s: pd.qcut(s, q=num_bins, duplicates="drop").map(lambda x: x.right if pd.notna(x) else 0))
)

#### Compression Forces

We will calculate the compression forces in a very similar way. However, we will ignore the resisting moment.

The live load, $L$, is $40$ psf, while the snow load, $S$, is $25$ psf and applies only to corridor walls as that is how the roof trusses are supported.

$$
C_{seismic} = \frac{M_{OT}}{l_{wall} - 0.5}\quad\text{(units in feet)}
$$

In [86]:
shear_walls["compression force"] = shear_walls["OTM"] / (shear_walls["wall length"] - wall_arm_shortening_length)
shear_walls["binned compression force"] = (
    shear_walls["compression force"]
    .groupby(level="Level")
    .transform(lambda s: pd.qcut(s, q=num_bins).map(lambda x: x.right if pd.notna(x) else 0))
    # .transform(lambda s: pd.qcut(s, q=num_bins, duplicates="drop").map(lambda x: x.right if pd.notna(x) else pd.NA))
)

#### Holddown Forces Table

With the tension and compression forces calculated, we can construct a table of the forces for each level. We will default to 5 bins for each level to give us options A through E.

However, there is only one table in the plans for all the holddown forces, so these will all get compared together as an aggregate of all buildings and simplified further.

In [ ]:
NUM_OPTIONS = 5
LABELS = list("ABCDE")

schedule_rows = []


def assign_schedule(group):
    group = group.copy()

    level = group.index.get_level_values("Level")[0]

    n = len(group)
    group["_bin"] = np.floor(np.arange(n) * min(NUM_OPTIONS, n) / n)

    schedule = (
        group.groupby("_bin")
        .agg(
            tension=("tension force", "max"),
            compression=("compression force", "max"),
        )
        .reset_index()
    )

    # convert to kips
    schedule["tension"] /= 1000
    schedule["compression"] /= 1000

    # sort ONLY schedule
    schedule["governing"] = schedule[["tension", "compression"]].max(axis=1)
    schedule = schedule.sort_values("governing").reset_index(drop=True)

    # assign A–E
    schedule["option"] = LABELS[: len(schedule)]

    # IMPORTANT: keep level in schedule
    schedule["Level"] = level

    # map bins → options
    bin_to_option = dict(zip(schedule["_bin"], schedule["option"]))
    group["option"] = group["_bin"].map(bin_to_option)

    schedule_map = schedule.set_index("option")[["tension", "compression"]]

    group["binned tension force"] = group["option"].map(schedule_map["tension"])
    group["binned compression force"] = group["option"].map(schedule_map["compression"])

    schedule_rows.append(schedule[["Level", "option", "tension", "compression"]])

    return group.drop(columns=["_bin"])


shear_walls = shear_walls.groupby(level="Level", group_keys=False).apply(assign_schedule)

schedule_df = (
    pd.concat(schedule_rows, ignore_index=True)
    .rename(
        columns={
            "tension": "binned tension force",
            "compression": "binned compression force",
        }
    )
    .set_index(["Level", "option"])
    .sort_index()
)

binned tension force  binned compression force
Level   option                                                
Level 2 A                  34.178228                 38.044389
        B                  51.328255                 51.328255
        C                  63.601702                 63.601702
        D                  72.616334                 72.616334
        E                  73.936631                 73.936631
Level 3 A                  21.740047                 25.606208
        B                  35.174430                 35.174430
        C                  44.076279                 44.076279
        D                  50.323461                 50.323461
        E                  51.238433                 51.238433
Level 4 A                  12.543913                 16.410074
        B                  21.494608                 21.494608
        C                  27.668600                 27.668600
        D                  31.590229                 31.590229
        E                  32.164597                 32.164597
Level 5 A                   4.529836                  8.210465
        B                  10.666458                 10.666458
        C                  13.961333                 13.961333
        D                  15.940152                 15.940152
        E                  16.229973                 16.229973
Roof    A                   0.000000                  2.490340
        B                   3.272601                  3.272601
        C                   4.243308                  4.243308
        D                   4.844736                  4.844736
        E                   4.932822                  4.932822

In [ ]:
column_name_map = {
    "binned tension force": "$T$ [kip]",
    "binned compression force": "$C$ [kip]",
}
display_table(
    schedule_df,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Holddown Schedule",
    label="tab:holddown_schedule",
)

## Diaphragm Design

We must first get the diaphragm inertial design forces. These are distinct from the seismic forces determined in \autoref{tab:seismic_forces_calculation_table}.

\begin{align*}
F_{px} &= \frac{\sum_{i=x}^n F_i}{\sum_{i=x}^n w_i} w_{px} \quad \text{(ASCE Eq. 12.10-1)} \\
F_{px} &\ge 0.2 S_{DS} I_e w_{px} \quad \text{(ASCE Eq. 12.10-2)} \\
F_{px} &\le 0.4 S_{DS} I_e w_{px}  \quad \text{(ASCE Eq. 12.10-3)}
\end{align*}

These forces are shown in \autoref{tab:diaphragm_forces}

In [88]:
seismic_loads["cumulative floor weight"] = seismic_loads.iloc[::-1]["floor weight"].cumsum()
seismic_loads["F_x,cum"] = seismic_loads.iloc[::-1]["F_x"].cumsum()
seismic_loads["F_px lower bound"] = 0.2 * S_DS * I_e * seismic_loads["floor weight"]
seismic_loads["F_px upper bound"] = 0.4 * S_DS * I_e * seismic_loads["floor weight"]
seismic_loads["F_px initial"] = (
    seismic_loads["F_x,cum"] / seismic_loads["cumulative floor weight"] * seismic_loads["floor weight"]
)

# Assign F_px with limits from ASCE
seismic_loads["F_px"] = np.where(
    seismic_loads["F_px initial"] <= seismic_loads["F_px lower bound"],
    seismic_loads["F_px lower bound"],
    np.where(
        seismic_loads["F_px initial"] >= seismic_loads["F_px upper bound"],
        seismic_loads["F_px upper bound"],
        seismic_loads["F_px initial"],
    ),
)

In [89]:
column_name_map = {
    "floor weight": "$w_{px}$ [kip]",
    "cumulative floor weight": r"$\sum w_x$ [kip]",
    "F_x,cum": r"$\sum F_x$ [kip]",
    "F_px initial": "$F_{px,init}$ [kip]",
    "F_px lower bound": "$F_{px,lower}$ [kip]",
    "F_px upper bound": "$F_{px,upper}$ [kip]",
    "F_px": "$F_{px}$ [kip]",
}
display_table(
    dataframe=seismic_loads.iloc[:0:-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Forces",
    label="tab:diaphragm_forces",
)

,$w_{px}$ [kip],$\sum w_x$ [kip],$\sum F_x$ [kip],"$F_{px,init}$ [kip]","$F_{px,lower}$ [kip]","$F_{px,upper}$ [kip]",$F_{px}$ [kip]
Level,,,,,,,
Roof,488.887917,488.887917,87.130707,87.130707,66.214484,132.428968,87.130707
Level 5,711.736583,1200.624500,190.044554,112.659421,96.396882,192.793763,112.659421
Level 4,713.836083,1914.460583,268.057724,99.949447,96.681236,193.362472,99.949447
Level 3,713.836083,2628.296667,320.866639,87.146245,96.681236,193.362472,96.681236
Level 2,718.035083,3346.331750,348.633678,74.807649,97.249944,194.499888,97.249944


With these diaphragm forces, we can design the diaphragm for the worst-case cantilever and simply-supported conditions.

The longest simply-supported span in the N-S direction is about 34.5 feet, while the longest cantilever span is about 32 feet. The depth of the diaphragm in the N-S direction is 67 feet. The longest simply-supported span in the E-W direction is about 5.5 feet, while the longest cantilever span is about 31 feet. The depth of the diaphragm in the E-W direction is 230 feet.

Diaphragm unit shear demands are shown in \autoref{tab:diaphragm_unit_shears}

In [ ]:
# Set up diaphragm dataframe
levels = ["Level 5", "Level 4", "Level 3", "Level 2", "Level 1"]
directions = ["N-S", "E-W"]
index = pd.MultiIndex.from_product(
    [levels, directions],
    names=["Level", "Direction"],
)
diaphragms = pd.DataFrame(index=index)
diaphragms_ns = diaphragms.xs("N-S", level="Direction").copy()
diaphragms_ew = diaphragms.xs("E-W", level="Direction").copy()

In [91]:
diaphragms_ns["F_px"] = seismic_loads["F_px"] * 1000
diaphragms_ns["area load"] = diaphragms_ns["F_px"] / float(A_level)
diaphragms_ns["L_simp"] = 35
diaphragms_ns["L_cant"] = 35
diaphragms_ns["depth"] = 67
diaphragms_ns["V_simp"] = diaphragms_ns["area load"] * diaphragms_ns["depth"] * diaphragms_ns["L_simp"] / 2
diaphragms_ns["V_cant"] = diaphragms_ns["area load"] * diaphragms_ns["depth"] * diaphragms_ns["L_cant"]
diaphragms_ns["v_simp"] = diaphragms_ns["V_simp"] / diaphragms_ns["depth"]
diaphragms_ns["v_cant"] = diaphragms_ns["V_cant"] / diaphragms_ns["depth"]

We will assume the diaphragm sheathing has the following properties:

- Unblocked
- WSP Sheathing
- 15/32" thick
- 10d common nails
- Plywood
- Nailed face is 2"
- Edge adjoining cases 2 through 6

The diaphragm capacity checks for this chosen sheathing are shown in \autoref{tab:diaphragm_capacities}

In [92]:
# Set baseline sheathing properties. We set the number of sheathed sides later
from structural_tools.seismic.timber.sheathing import SheathingApplication

diaphragm_sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
    minimum_nominal_width_of_nailed_face=2,
    adjoining_panel_edge_location=(2, 3, 4, 5, 6),
)

diaphragms_ns[
    ["adjusted unit shear capacity simply supported", "shear stiffness simply supported", "shear dcr simply supported"]
] = get_sheathing_properties(
    diaphragms_ns["v_simp"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
)

diaphragms_ns[["adjusted unit shear capacity cantilever", "shear stiffness cantilever", "shear dcr cantilever"]] = (
    get_sheathing_properties(
        diaphragms_ns["v_cant"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
    )
)

In [93]:
diaphragms_ew["F_px"] = seismic_loads["F_px"] * 1000
diaphragms_ew["area load"] = diaphragms_ew["F_px"] / float(A_level)
diaphragms_ew["L_simp"] = 25
diaphragms_ew["L_cant"] = 35
diaphragms_ew["depth"] = 67
diaphragms_ew["V_simp"] = diaphragms_ew["area load"] * diaphragms_ew["depth"] * diaphragms_ew["L_simp"] / 2
diaphragms_ew["V_cant"] = diaphragms_ew["area load"] * diaphragms_ew["depth"] * diaphragms_ew["L_cant"]
diaphragms_ew["v_simp"] = diaphragms_ew["V_simp"] / diaphragms_ew["depth"]
diaphragms_ew["v_cant"] = diaphragms_ew["V_cant"] / diaphragms_ew["depth"]


In [94]:
diaphragms_ew[
    ["adjusted unit shear capacity simply supported", "shear stiffness simply supported", "shear dcr simply supported"]
] = get_sheathing_properties(
    diaphragms_ew["v_simp"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
)

diaphragms_ew[["adjusted unit shear capacity cantilever", "shear stiffness cantilever", "shear dcr cantilever"]] = (
    get_sheathing_properties(
        diaphragms_ew["v_cant"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
    )
)

In [95]:
idx = pd.IndexSlice
diaphragms = diaphragms.reindex(columns=diaphragms_ns.columns)
diaphragms_ns["Direction"] = "N-S"
diaphragms_ew["Direction"] = "E-W"
diaphragms_ns = diaphragms_ns.set_index("Direction", append=True)
diaphragms_ew = diaphragms_ew.set_index("Direction", append=True)
diaphragms.update(diaphragms_ns)
diaphragms.update(diaphragms_ew)

In [96]:
column_name_map = {
    "F_px": "$F_{px}$ [lbf]",
    "area load": "$w$ [psf]",
    "V_simp": "$V_{simp}$ [lbf]",
    "V_cant": "$V_{cant}$ [lbf]",
    "v_simp": "$v_{simp}$ [plf]",
    "v_cant": "$v_{cant}$ [plf]",
}
display_table(
    dataframe=diaphragms,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Unit Shear Demands",
    label="tab:diaphragm_unit_shears",
)

$F_{px}$ [lbf]  $w$ [psf]  $V_{simp}$ [lbf]  \
Level   Direction                                                
Roof    N-S          87130.707315   4.874195       5714.993879   
        E-W          87130.707315   4.874195       4082.138485   
Level 5 N-S         112659.421436   6.302302       7389.448838   
        E-W         112659.421436   6.302302       5278.177741   
Level 4 N-S          99949.446523   5.591291       6555.788340   
        E-W          99949.446523   5.591291       4682.705957   
Level 3 N-S          96681.235773   5.408463       6341.422991   
        E-W          96681.235773   5.408463       4529.587851   
Level 2 N-S          97249.944078   5.440277       6378.725162   
        E-W          97249.944078   5.440277       4556.232258   

                   $V_{cant}$ [lbf]  $v_{simp}$ [plf]  $v_{cant}$ [plf]  
Level   Direction                                                        
Roof    N-S            11429.987758         85.298416        170.596832  
        E-W            11429.987758         60.927440        170.596832  
Level 5 N-S            14778.897675        110.290281        220.580562  
        E-W            14778.897675         78.778772        220.580562  
Level 4 N-S            13111.576680         97.847587        195.695174  
        E-W            13111.576680         69.891134        195.695174  
Level 3 N-S            12682.845983         94.648104        189.296209  
        E-W            12682.845983         67.605789        189.296209  
Level 2 N-S            12757.450324         95.204853        190.409706  
        E-W            12757.450324         68.003467        190.409706

In [97]:
column_name_map = {
    "v_simp": "$v_{simp}$ [plf]",
    "adjusted unit shear capacity simply supported": "$v_{cap,simp}$ [plf]",
    "shear dcr simply supported": "DCR$_{simp}$ [-]",
    "v_cant": "$v_{cant}$ [plf]",
    "adjusted unit shear capacity cantilever": "$v_{cap,cant}$ [plf]",
    "shear dcr cantilever": "DCR$_{cant}$ [-]",
}
display_table(
    dataframe=diaphragms,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Capacity Checks",
    label="tab:diaphragm_capacities",
)

$v_{simp}$ [plf]  $v_{cap,simp}$ [plf]  DCR$_{simp}$ [-]  \
Level   Direction                                                             
Roof    N-S               85.298416            189.285714          0.450633   
        E-W               60.927440            189.285714          0.321881   
Level 5 N-S              110.290281            189.285714          0.582666   
        E-W               78.778772            189.285714          0.416190   
Level 4 N-S               97.847587            189.285714          0.516931   
        E-W               69.891134            189.285714          0.369236   
Level 3 N-S               94.648104            189.285714          0.500028   
        E-W               67.605789            189.285714          0.357163   
Level 2 N-S               95.204853            189.285714          0.502969   
        E-W               68.003467            189.285714          0.359264   

                   $v_{cant}$ [plf]  $v_{cap,cant}$ [plf]  DCR$_{cant}$ [-]  
Level   Direction                                                            
Roof    N-S              170.596832            189.285714          0.901266  
        E-W              170.596832            189.285714          0.901266  
Level 5 N-S              220.580562            255.357143          0.863812  
        E-W              220.580562            255.357143          0.863812  
Level 4 N-S              195.695174            214.285714          0.913244  
        E-W              195.695174            214.285714          0.913244  
Level 3 N-S              189.296209            214.285714          0.883382  
        E-W              189.296209            214.285714          0.883382  
Level 2 N-S              190.409706            214.285714          0.888579  
        E-W              190.409706            214.285714          0.888579

In [98]:
if (
    (diaphragms[["shear dcr simply supported", "shear dcr cantilever"]] <= 1).all().all()
):  # double .all() because 2 columns
    display_text(r"$\therefore$ All shear capacities are OK")
else:
    display_text("You must check for new diaphragm nailing.")

'$\\therefore$ All shear capacities are OK'

## Story Drift Determination

The inelastic design story drift is calculated as the difference in deflections between stories at their centers of mass.

We can find the shear wall line that is closest to the center of mass, then take the cumulative force on that line and divide it by the cumulative wall stiffness on that line. This will give us an approximate elastic story drift that will then be converted to inelastic story drift with $C_d$ and $I_e$.

Note that this is a conservative estimation of inelastic story drift due to assuming the deflection of a single shear wall line, rather than the full deflection of the diaphragm as a rigid unit.

\begin{align*}
    \delta_{xe} &= F/k \\
    \delta_x &= \frac{C_d \delta_{xe}}{I_e}
\end{align*}

In [ ]:
%%render
C_d
I_e

<IPython.core.display.Latex object>

In [ ]:
shear_walls["line"] = flexible["line"]
shear_walls["wall stiffness"] = rigid["wall stiffness"]
shear_walls["cumulative shear demand"] = flexible["cumulative shear demand"]
shear_walls["x"] = rigid["x"]
shear_walls["y"] = rigid["y"]
shear_walls["delta_sw"] = rigid["delta_sw"]


def line_deflection(group, line):
    line_walls = group[group["line"] == line]

    total_stiffness = line_walls["wall stiffness"].sum()
    total_shear = line_walls["cumulative shear demand"].sum()

    return total_shear / total_stiffness


def get_cm_line_deflection(group, CM_x, CM_y, C_d, I_e):
    x_wall = group.loc[(group["x"] - CM_x).abs().idxmin()]

    y_wall = group.loc[(group["y"] - CM_y).abs().idxmin()]

    x_line = x_wall["line"]
    y_line = y_wall["line"]

    x_delta_sw = line_deflection(group, x_line)
    y_delta_sw = line_deflection(group, y_line)

    drift_factor = C_d / I_e

    return pd.Series(
        {
            "x line": x_line,
            "x delta_sw": x_delta_sw,
            "x inelastic drift": x_delta_sw * drift_factor,
            "y line": y_line,
            "y delta_sw": y_delta_sw,
            "y inelastic drift": y_delta_sw * drift_factor,
        }
    )


story_drift = shear_walls.groupby(level="Level", sort=False).apply(
    get_cm_line_deflection,
    CM_x=CM_x,
    CM_y=CM_y,
    C_d=C_d,
    I_e=I_e,
)
story_drift["allowable story drift"] = seismic_loads["floor height"] * 0.02 * 12

In [ ]:
column_name_map = {
    "x line": "N-S line",
    "x delta_sw": r"$\delta_{xe,ns}$ [in]",
    "x inelastic drift": r"$\delta_{x,ns}$ [in]",
    "y line": "E-W line",
    "y delta_sw": r"$\delta_{xe,ew}$ [in]",
    "y inelastic drift": r"$\delta_{x,ew}$ [in]",
    "allowable story drift": r"$\Delta_a$ [in]",
}
display_table(
    story_drift,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Story Drifts",
    label="tab:story_drifts",
)

,N-S line,"$\delta_{sw,ns}$ [in]","$\delta_{x,ns}$ [in]",E-W line,"$\delta_{sw,ew}$ [in]","$\delta_{x,ew}$ [in]",$\Delta_a$ [in]
Level,,,,,,,
Level 4,A-6,0.197989,0.791958,A-B,0.374226,1.496904,2.40
Level 3,A-6,0.341671,1.366682,A-B,0.565314,2.261256,2.52
Level 2,A-6,0.373059,1.492237,A-B,0.653255,2.613021,2.52
Level 1,A-6,0.453864,1.815455,A-B,0.697459,2.789837,2.76


In [ ]:
# Save dataframes to CSV
shear_walls.to_csv("shear_walls_E.csv")